<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase2/phase2_03_labelled_frame_manifest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Sector 3: Clinical Taxonomy, Labelled-Frame Manifest, and Class Coverage

This notebook consumes the saved Sector 1 metadata and Sector 2 alignment/decode audits. It builds a portable annotation-level manifest, validates clinical-label mappings and decoded-frame bounds, evaluates bounding boxes, and reports class coverage before and after alignment filtering. The binary video/class matrix is ready for downstream video-level stratification; this notebook does not assign splits.

Configuration is inherited from both upstream notebooks and their available runtime snapshots, without executing their code. Pure DF transformations are separated from explicit I/O functions. No video decoding, image loading, GPU dependency, or model training is needed.



### 1. Minimal dependencies and imports

Only table processing, Parquet, and notebook display dependencies are required. Git is supplied by Colab.


In [16]:
%pip install -q pandas numpy pyarrow


In [17]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import pandas as pd
import ast
import copy
import hashlib
import json
import os
import re
import sys
import subprocess
import numpy as np
from pathlib import Path, PurePosixPath
import contextlib
import shutil
import tempfile
from IPython.display import display
import math
from types import CodeType, FunctionType


### 2. Declarative bootstrap and Sector 3 policy



In [18]:
BOOTSTRAP = {
    "repo_url": "https://github.com/bogdanparvu18/msc-graduate-project.git",
    "branch": "main",
    "local_repo_dir": "/content/msc-graduate-project",
    "notebook_paths": {
        "sector1": "notebooks/phase2/phase2_01_ingestion.ipynb",
        "sector2": "notebooks/phase2/phase2_02_video_frame_validation.ipynb",
    },
    "storage_root": None,
    "sector1_config_path": None,
    "sector2_config_path": None,
    "allow_local_test_remote": False,
}

SECTOR3_SETTINGS = {
    "local_output_dir": "outputs/phase2/sector3",
    "export_dir": "outputs/phase2/sector3",
    "require_any_bbox": True,
    "require_valid_bbox": True,
    "config_overrides": {},
    "push": True,
    "dry_run": False,
}


### 3. Configuration inheritance and validated input loading



In [19]:
# ------------------------------------------------------------------------------
# Execution metadata is retained in saved envelopes, but does not define content.
# ------------------------------------------------------------------------------

S3_EXECUTION_METADATA_KEYS = frozenset({
    "run_id", "run_context", "run_datetime", "run_timestamp", "run_date", "run_time",
    "timestamp", "date", "time", "execution_date", "execution_time",
    "created_at", "created_at_utc", "created_utc", "saved_at", "saved_at_utc", "saved_utc",
    "started_at", "started_at_utc", "started_utc", "finished_at", "finished_at_utc", "finished_utc",
    "generated_at", "generated_at_utc", "generated_utc", "updated_at", "updated_at_utc", "updated_utc",
    "modified_at", "modified_at_utc", "modified_utc",
    "source_commit", "source_commit_sha", "source_commit_hash", "source_revision", "config_source_commit",
    "commit", "commit_sha", "head_commit", "current_commit", "repository_commit",
    "publication_receipt", "receipt", "last_publication",
    "source_configs", "config_provenance", "config_conflict_report", "lineage",
    "source_notebooks", "source_config_provenance", "config_sources", "literal_notebooks", "literal_configs",
    "input_sha256", "state_sha256", "upstream_states", "upstream_alignment_run_id",
    "notebook_path", "notebook_sha256", "source_notebook_path", "source_notebook_sha256",
    "source_config_path", "source_config_sha256", "source_snapshot_path", "source_snapshot_sha256",
    "config_snapshot_path", "config_snapshot_sha256", "snapshot_path", "snapshot_sha256",
    "source_config_file", "source_config_hash", "source_snapshot_file", "source_snapshot_hash",
    "config_snapshot_file", "config_snapshot_hash", "snapshot_file", "snapshot_hash",
    "config_inheritance_policy", "taxonomy_validation_policy",
})

S3_CONSUMED_INPUT_FILENAMES = (
    "metadata_clean.parquet", "frame_alignment_confirmed.parquet", "video_manifest.csv", "decode_audit.csv",
)


def _s3_semantic_json_value(value):
    """Pure conversion to canonical JSON-compatible values; no I/O or clock access."""
    if value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, dict):
        return {str(key): _s3_semantic_json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_s3_semantic_json_value(item) for item in value]
    if isinstance(value, (set, frozenset)):
        values = [_s3_semantic_json_value(item) for item in value]
        return sorted(values, key=lambda item: json.dumps(item, sort_keys=True))
    if isinstance(value, (Path, PurePosixPath)):
        return str(value)
    if isinstance(value, np.generic):
        return _s3_semantic_json_value(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        raise ValueError("Semantic configuration must not contain NaN or infinite values.")
    if value is None or isinstance(value, (str, bool, int, float)):
        return value
    raise TypeError(f"Unsupported semantic configuration value: {type(value).__name__}")


def _s3_semantic_json(value):
    return json.dumps(_s3_semantic_json_value(value), ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def _s3_semantic_digest(value):
    return hashlib.sha256(_s3_semantic_json(value).encode("utf-8")).hexdigest()


def sector3_semantic_config(config):
    """Project authored effective values, excluding execution/source bookkeeping only.

    Full source snapshots and current execution metadata remain available in the
    saved envelope. Authored paths, thresholds, publication settings (including
    push/dry_run), and unknown substantive config keys retain their meaning.
    """
    if not isinstance(config, dict):
        raise TypeError("Sector 3 semantic configuration requires a dictionary.")
    def project(value):
        if isinstance(value, dict):
            return {str(key): project(item) for key, item in value.items()
                    if str(key).lower() not in S3_EXECUTION_METADATA_KEYS}
        if isinstance(value, (list, tuple)):
            return [project(item) for item in value]
        return _s3_semantic_json_value(value)
    return project(config)


def sector3_semantic_config_fingerprint(config):
    """Stable identity of authored configuration, independent of execution time."""
    return _s3_semantic_digest(sector3_semantic_config(config))


def semantic_config_fingerprint(config):
    """Compatibility alias used by the persistence layer."""
    return sector3_semantic_config_fingerprint(config)


def _s3_semantic_conflict_value(value):
    if value is pd.NA or value is None:
        return "null"
    if isinstance(value, str):
        if value == "<not declared>":
            return value
        try:
            return _s3_semantic_json(json.loads(value))
        except (json.JSONDecodeError, TypeError):
            return _s3_semantic_json(value)
    return _s3_semantic_json(value)


def sector3_semantic_conflict_report(report):
    """Return a deterministic report of authored source differences only."""
    columns = ["config_key", "sector1_value", "sector2_value", "resolution"]
    if not isinstance(report, pd.DataFrame) or not set(columns).issubset(report.columns):
        raise ValueError("Configuration conflict reports require the four declared report columns.")
    keys = report["config_key"].astype("string")
    if keys.isna().any():
        raise ValueError("Configuration conflict keys must be present.")
    escaped = "|".join(re.escape(key) for key in sorted(S3_EXECUTION_METADATA_KEYS))
    runtime = keys.str.lower().str.contains(r"(?:^|\.)(?:" + escaped + r")(?:\.|$)", regex=True, na=False)
    result = report.loc[~runtime, columns].copy()
    for column in ("sector1_value", "sector2_value"):
        result[column] = result[column].map(_s3_semantic_conflict_value)
    return (result.astype("string").sort_values(columns, kind="stable")
            .drop_duplicates().reset_index(drop=True))


def sector3_processing_config(config):
    """Pure cache projection of the parameters read by the current core functions.

    Publication and cache controls do not cause re-transformation. This selector
    must be updated whenever the core starts reading another parameter; the exact
    active core fingerprint also invalidates old transformation caches after code edits.
    """
    settings = config.get("sector3", {})
    references = config.get("decode_audit_published_reference_frames", {})
    return _s3_semantic_json_value({
        "storage_root": config["storage_root"],
        "source_storage_root": config.get("source_storage_root", config["storage_root"]),
        "clinical_group_map": config.get("clinical_group_map"),
        "expected_total_extractable_frames": config.get("expected_total_extractable_frames", references.get("all_videos")),
        "expected_labelled_frames": config.get("expected_labelled_frames"),
        "decode_audit_published_reference_frames": references,
        "frame_alignment_min_ssim": config.get("frame_alignment_min_ssim"),
        "sector3": {
            "require_any_bbox": settings.get("require_any_bbox", True),
            "require_valid_bbox": settings.get("require_valid_bbox", True),
            "rare_video_support_threshold": settings.get("rare_video_support_threshold", 3),
        },
    })


def sector3_cache_signature(input_sha256, core_source_sha256, config):
    """Identify transformations by consumed input bytes, core code, and core params."""
    if not isinstance(input_sha256, dict):
        raise TypeError("Input SHA-256 identities must be supplied as a dictionary.")
    if not isinstance(core_source_sha256, str) or not re.fullmatch(r"[0-9a-f]{64}", core_source_sha256):
        raise ValueError("A SHA-256 fingerprint of the active pure core is required.")
    selected = {}
    for path, digest in input_sha256.items():
        filename = PurePosixPath(str(path)).name
        if filename not in S3_CONSUMED_INPUT_FILENAMES:
            continue
        if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):
            raise ValueError(f"A valid input SHA-256 is required for {filename}.")
        if filename in selected and selected[filename] != digest:
            raise ValueError(f"Conflicting byte identities were supplied for {filename}.")
        selected[filename] = digest
    missing = sorted(set(S3_CONSUMED_INPUT_FILENAMES) - set(selected))
    if missing:
        raise ValueError(f"The consumed Sector 3 input hashes are incomplete: {missing}")
    return _s3_semantic_digest({"input_sha256": selected, "core_source_sha256": core_source_sha256,
                                "processing_config": sector3_processing_config(config)})


In [20]:
def io_s3_git(repo, *arguments):
    """Run Git without a shell or embedded credentials."""
    environment = {key: value for key, value in os.environ.items()
                   if not key.startswith("GIT_TRACE") and key != "GIT_CURL_VERBOSE"}
    environment["GIT_TERMINAL_PROMPT"] = "0"
    command = ["git", *(["-C", str(repo)] if repo is not None else []), *arguments]
    process = subprocess.run(command, env=environment, capture_output=True, text=True)
    if process.returncode:
        raise RuntimeError("Git failed: " + (process.stderr or process.stdout).strip())
    return process.stdout.strip()


def io_bootstrap_repository(bootstrap):
    """Clone once; fetch and fast-forward a clean checkout of the declared branch."""
    repo = Path(bootstrap["local_repo_dir"]).expanduser().resolve()
    branch, url = bootstrap["branch"], bootstrap["repo_url"]
    is_github = re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?", url)
    is_local_test = bootstrap.get("allow_local_test_remote") is True and Path(url).is_absolute() and Path(url).is_dir()
    if not is_github and not is_local_test:
        raise ValueError("Bootstrap repo_url must be a credential-free GitHub HTTPS URL.")
    if not (repo / ".git").is_dir():
        if repo.exists() and (not repo.is_dir() or any(repo.iterdir())):
            raise RuntimeError("The bootstrap directory exists and is not an empty Git checkout.")
        repo.parent.mkdir(parents=True, exist_ok=True)
        io_s3_git(None, "clone", "--single-branch", "--branch", branch, url, str(repo))
    expected = url.rstrip("/").removesuffix(".git")
    for options in (("get-url",), ("get-url", "--push")):
        if io_s3_git(repo, "remote", *options, "origin").rstrip("/").removesuffix(".git") != expected:
            raise RuntimeError("Origin does not match the declared repository.")
    if io_s3_git(repo, "branch", "--show-current") != branch:
        raise RuntimeError("The checkout branch differs from bootstrap; no automatic switch is made.")
    if io_s3_git(repo, "status", "--porcelain"):
        raise RuntimeError("The bootstrap checkout has changes; resolve them before fetching.")
    io_s3_git(repo, "fetch", "origin", branch)
    if io_s3_git(repo, "rev-list", f"origin/{branch}..HEAD"):
        raise RuntimeError("Unpublished local commits exist; resolve them before a new Sector 3 run.")
    io_s3_git(repo, "merge", "--ff-only", f"origin/{branch}")
    return repo


def io_s3_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def parse_upstream_literals(notebook):
    """Pure AST parsing: never execute imports, cells, assignments, or magics."""
    values = {"CONFIG": [], "GIT_CONFIG": []}
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", "")
        source = "".join(source) if isinstance(source, list) else source
        try:
            statements = ast.parse(source).body
        except SyntaxError:
            if re.search(r"(?m)^\s*(?:CONFIG|GIT_CONFIG)\s*=\s*\{", source):
                raise ValueError("Literal config cells must contain plain Python without magics.") from None
            continue
        for statement in statements:
            targets = statement.targets if isinstance(statement, ast.Assign) else (
                [statement.target] if isinstance(statement, ast.AnnAssign) else [])
            names = [target.id for target in targets if isinstance(target, ast.Name) and target.id in values]
            if names and isinstance(statement.value, ast.Dict):
                try:
                    literal = ast.literal_eval(statement.value)
                    json.dumps(literal, allow_nan=False)
                except (ValueError, TypeError, SyntaxError):
                    raise ValueError("CONFIG and GIT_CONFIG must use literal JSON-compatible values.") from None
                for name in names:
                    values[name].append(literal)
    if any(len(found) != 1 for found in values.values()):
        raise ValueError("Each upstream notebook must declare exactly one literal CONFIG and GIT_CONFIG.")
    return {name: found[0] for name, found in values.items()}


def io_load_upstream_notebooks(repo, paths):
    """Read both declared source notebooks and record the tracked checkout identity."""
    repo = Path(repo).resolve()
    loaded = {}
    for sector in ("sector1", "sector2"):
        relative = Path(paths[sector])
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Upstream notebook paths must be repository-relative.")
        path = repo / relative
        if not path.is_file():
            raise FileNotFoundError(f"The upstream notebook is missing: {relative}")
        loaded[sector] = {**parse_upstream_literals(json.loads(path.read_text(encoding="utf-8"))),
                          "notebook_path": relative.as_posix(), "notebook_sha256": io_s3_sha256(path)}
    return {**loaded, "source_commit": io_s3_git(repo, "rev-parse", "HEAD")}


def sector3_config_conflicts(source1, source2, prefix=""):
    """Pure declaration comparison, including added or inherited source keys."""
    records = []
    for key in sorted(set(source1) | set(source2)):
        path = f"{prefix}.{key}" if prefix else key
        if key not in source1 or key not in source2:
            records.append({"config_key": path,
                            "sector1_value": json.dumps(source1[key], sort_keys=True) if key in source1 else "<not declared>",
                            "sector2_value": json.dumps(source2[key], sort_keys=True) if key in source2 else "<not declared>",
                            "resolution": "sector2_runtime_added" if key in source2 else "sector1_inherited"})
            continue
        left, right = source1[key], source2[key]
        if isinstance(left, dict) and isinstance(right, dict):
            records.extend(sector3_config_conflicts(left, right, path))
        elif left != right:
            records.append({"config_key": path,
                            "sector1_value": json.dumps(left, sort_keys=True),
                            "sector2_value": json.dumps(right, sort_keys=True),
                            "resolution": "sector2_runtime_precedence"})
    return records


def merge_sector3_config_values(source1, source2):
    """Pure recursive copy: Sector 1 supplies base values, Sector 2 supplies updates."""
    merged = copy.deepcopy(source1)
    for key, value in source2.items():
        merged[key] = merge_sector3_config_values(merged[key], value) if (
            isinstance(merged.get(key), dict) and isinstance(value, dict)) else copy.deepcopy(value)
    return merged


def build_sector3_config(source1, source2, sector3_settings, git_source, run_context, provenance):
    """Pure config inheritance; source snapshots are retained, overrides are explicit."""
    overrides = sector3_settings.get("config_overrides", {})
    result = merge_sector3_config_values(merge_sector3_config_values(source1, source2), overrides)
    conflicts = sector3_config_conflicts(source1, source2)
    settings = {"export_dir": "outputs/phase2/sector3", "local_output_dir": "outputs/phase2/sector3",
                "require_any_bbox": True, "require_valid_bbox": True, **copy.deepcopy(sector3_settings)}
    github = {**copy.deepcopy(git_source), "repo_output_dir": settings["export_dir"],
              "push": settings.get("push", True), "dry_run": settings.get("dry_run", False),
              "commit_message": "Phase 2 Sector 3: publish labelled-frame manifest and clinical audits"}
    github.pop("confirm_push", None)
    github.update(copy.deepcopy(settings.get("github_overrides", {})))
    return {**result, **copy.deepcopy(run_context), "phase": 2, "sector": 3,
            "source_storage_root": source2["storage_root"],
            "source_storage_roots": {"sector1": source1["storage_root"], "sector2": source2["storage_root"]},
            "storage_root": provenance["runtime_storage_root"],
            "storage_backend": provenance.get("runtime_storage_backend", result.get("storage_backend")),
            "sector3": settings, "github": github,
            "source_configs": {"sector1": copy.deepcopy(source1), "sector2": copy.deepcopy(source2),
                               "literal_notebooks": copy.deepcopy(provenance["literal_configs"])},
            "config_conflict_report": conflicts, "config_provenance": copy.deepcopy(provenance),
            "config_inheritance_policy": "Sector 1 base; Sector 2 runtime precedence; declared config_overrides last.",
            "taxonomy_validation_policy": "Validate the effective map against source and confirmed annotation groups; preserve original labels."}


def sector3_resolved_input_dirs(config, runtime_root=None):
    """Pure lexical resolution, including explicit storage-root rebasing."""
    source_root = Path(config["storage_root"])
    current_root = Path(runtime_root or source_root)
    def resolve(value):
        path = Path(value)
        if not path.is_absolute():
            return current_root / path
        if current_root != source_root:
            try:
                return current_root / path.relative_to(source_root)
            except ValueError:
                raise ValueError("An upstream absolute input directory lies outside its source storage root.") from None
        return path
    output = resolve(config["output_dir"])
    curated = resolve(config["curated_data_dir"])
    return {"configs": output / "configs", "results": resolve(config["results_dir"]) if "results_dir" in config else output / "results",
            "manifests": curated / "manifests", "metadata": curated / "dataset_audit/metadata_clean.parquet"}


def io_s3_read_json(path):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"Expected a JSON object: {path}")
    return payload


def io_s3_config_candidates(repo, literal, sector, bootstrap):
    """Saved Drive snapshots are preferred; repository snapshots are provenance-aware fallbacks."""
    explicit = bootstrap.get(f"{sector}_config_path")
    if explicit:
        paths = [Path(explicit)]
    else:
        directory = sector3_resolved_input_dirs(literal, bootstrap.get("storage_root"))["configs"]
        stem = "phase2_01_ingestion" if sector == "sector1" else "phase2_02_video_frame_validation"
        directories = [directory, Path(repo) / "outputs/phase2" / sector / "configs", Path(repo) / "outputs/phase2/configs"]
        paths = [path for parent in directories for path in sorted(parent.glob(stem + "_*_config.json"), reverse=True)]
    return [(path, io_s3_read_json(path)) for path in dict.fromkeys(paths)]


def io_s3_checked_state(path):
    state = io_s3_read_json(path)
    if state.get("complete") is not True:
        raise RuntimeError(f"Upstream processing is incomplete: {path}")
    return state


def io_s3_check_hash(path, expected):
    if not isinstance(expected, str) or not re.fullmatch(r"[0-9a-f]{64}", expected):
        raise ValueError(f"A valid upstream SHA-256 is required for {path}")
    actual = io_s3_sha256(path)
    if actual != expected:
        raise ValueError(f"Upstream artifact changed after its completed audit: {path}")
    return actual


def validate_sector3_input_partition(metadata, confirmed, excluded, states, manifest, decode):
    """Pure count, identity, accepted-subset, and completed-video contract checks."""
    alignment = states["alignment"]
    expected = (alignment.get("original_annotation_rows"), alignment.get("confirmed_annotation_rows"),
                alignment.get("excluded_annotation_rows"))
    if expected != (len(metadata), len(confirmed), len(excluded)) or len(confirmed) + len(excluded) != len(metadata):
        raise ValueError("The completed alignment counts do not partition the original cleaned metadata.")
    identities = [column for column in ("filename", "image_key", "video_key", "frame_number",
                  "finding_class_normalized", "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax") if column in metadata]
    if not {"video_key", "frame_number", "finding_class_normalized"}.issubset(identities):
        raise KeyError("Source metadata lacks the annotation identity fields required to verify the partition.")
    def row_counts(table):
        if not set(identities).issubset(table.columns):
            raise KeyError("The alignment subset lacks original annotation identity fields.")
        canonical = table[identities].astype("string").fillna("")
        numeric = [column for column in identities if column == "frame_number" or column.startswith("bbox_")]
        canonical[numeric] = table[numeric].apply(pd.to_numeric, errors="coerce").astype("Float64").astype("string").fillna("")
        return canonical.value_counts(sort=False).sort_index()
    # Compare multiplicities after canonicalizing each table separately. This
    # avoids dtype inference from concatenating an empty/all-NA excluded table.
    source_counts = row_counts(metadata).astype("int64")
    subset_counts = row_counts(confirmed).add(row_counts(excluded), fill_value=0).astype("int64").sort_index()
    if not source_counts.equals(subset_counts):
        raise ValueError("Confirmed/excluded annotation identities do not exactly partition the source metadata.")
    keys = manifest["video_key"].astype("string")
    decode_keys = decode["video_key"].astype("string")
    if keys.isna().any() or keys.duplicated().any() or decode_keys.isna().any() or decode_keys.duplicated().any() or set(keys) != set(decode_keys):
        raise ValueError("The video manifest and decode audit must contain the same unique nonempty video keys.")
    if len(manifest) != states["manifest"].get("video_count") or len(decode) != states["decode"].get("video_count"):
        raise ValueError("Video table counts disagree with their completed upstream states.")
    if states["decode"].get("all_videos_decoded") is not True:
        raise ValueError("Finish decoding every source video before building the physical-frame inventory.")
    required = {"aligned_frame_index", "alignment_training_eligible", "video_key", "alignment_mapping_id"}
    if not required.issubset(confirmed.columns):
        raise KeyError("The confirmed metadata lacks the current Sector 2 alignment contract.")
    if not confirmed["alignment_training_eligible"].astype("string").str.lower().eq("true").all():
        raise ValueError("The confirmed metadata contains an annotation not accepted by alignment.")
    if not set(confirmed["video_key"].astype("string")).issubset(set(keys)):
        raise ValueError("Confirmed metadata references videos outside the audited inventory.")

def io_s3_load_current_alignment(alignment_path):
    """Read a completed attempt and the exact final state it committed."""
    alignment_path = Path(alignment_path)
    attempt_path = alignment_path.with_name(
        "frame_alignment_attempt_state.json"
    )

    try:
        final_bytes = alignment_path.read_bytes()
        attempt_bytes = attempt_path.read_bytes()
        alignment = json.loads(final_bytes)
        attempt = json.loads(attempt_bytes)
    except (OSError, ValueError) as error:
        raise RuntimeError(
            "A verifiable completed Sector 2 alignment is required."
        ) from error

    if not isinstance(alignment, dict) or not isinstance(attempt, dict):
        raise ValueError("Invalid Sector 2 alignment state objects.")

    state_hashes = {
        "alignment": hashlib.sha256(final_bytes).hexdigest(),
        "alignment_attempt": hashlib.sha256(attempt_bytes).hexdigest(),
    }

    attempt_id = attempt.get("attempt_id")
    run_id = alignment.get("run_id")

    if (
        alignment.get("complete") is not True
        or attempt.get("complete") is not True
        or attempt.get("status") != "completed"
        or not isinstance(attempt_id, str)
        or re.fullmatch(r"[0-9a-f]{32}", attempt_id) is None
        or alignment.get("attempt_id") != attempt_id
        or not isinstance(run_id, str)
        or not run_id.strip()
        or attempt.get("run_id") != run_id
        or attempt.get("final_state_sha256") != state_hashes["alignment"]
    ):
        raise RuntimeError(
            "The latest alignment attempt is incomplete, failed, or "
            "does not match the saved final audit. "
            "Complete Sector 2 before Sector 3."
        )

    return alignment, attempt, state_hashes

def io_load_sector3_context(repo, bootstrap, sector3_settings, notebook_sources, run_context):
    """Load saved source configs and complete, hash-verified upstream input artifacts."""
    literals = {sector: notebook_sources[sector]["CONFIG"] for sector in ("sector1", "sector2")}
    candidates = {sector: io_s3_config_candidates(repo, literals[sector], sector, bootstrap) for sector in literals}
    runtime_root = bootstrap.get("storage_root")
    initial = sector3_resolved_input_dirs(literals["sector2"], runtime_root)
    alignment_path = Path(bootstrap.get("alignment_state_path", initial["results"] / "frame_alignment_state.json"))
    if not alignment_path.is_file():
        raise FileNotFoundError("Run Sector 2 on mounted storage first, or declare the current alignment_state_path.")
    alignment, alignment_attempt, alignment_state_hashes = (io_s3_load_current_alignment(alignment_path))
    run_id = alignment.get("run_id")
    matching = [(path, config) for path, config in candidates["sector2"] if config.get("run_id") == run_id]
    if not matching:
        if run_id:
            raise FileNotFoundError(f"The completed alignment run {run_id} requires its exact saved Sector 2 config snapshot.")
        source2_path, source2 = None, literals["sector2"]
    else:
        source2_path, source2 = matching[0]
        if any(config != source2 for _, config in matching[1:]):
            raise ValueError("Multiple Sector 2 snapshots use the alignment run_id with differing configurations.")
    source1_path, source1 = candidates["sector1"][0] if candidates["sector1"] else (None, literals["sector1"])
    dirs1 = sector3_resolved_input_dirs(source1, runtime_root)
    dirs2 = sector3_resolved_input_dirs(source2, runtime_root)
    if alignment_path.resolve() != (dirs2["results"] / "frame_alignment_state.json").resolve():
        raise ValueError("The selected Sector 2 snapshot points at a different alignment state location.")
    states = {"alignment": alignment,
              "manifest": io_s3_checked_state(dirs2["manifests"] / "video_manifest_state.json"),
              "decode": io_s3_checked_state(dirs2["results"] / "decode_audit_state.json")}
    states["alignment_attempt"] = alignment_attempt
    thresholds = {"min_ssim": "frame_alignment_min_ssim", "min_margin": "frame_alignment_min_margin",
                  "candidate_offsets": "frame_index_offset_candidates"}
    if any(alignment.get("effective_settings", {}).get(recorded) != source2.get(configured)
           for recorded, configured in thresholds.items()):
        raise ValueError("The selected Sector 2 snapshot differs from the completed alignment decision settings.")
    hashes = {}
    for family in ("alignment", "decode"):
        output_hashes = states[family].get("output_sha256", {})
        if not output_hashes:
            raise ValueError(f"The {family} state has no committed output hashes.")
        for filename, expected in output_hashes.items():
            if Path(filename).name != filename:
                raise ValueError("Upstream states must list simple output filenames.")
            path = dirs2["results"] / filename
            hashes[str(path)] = io_s3_check_hash(path, expected)
    manifest_path = dirs2["manifests"] / "video_manifest.csv"
    hashes[str(manifest_path)] = io_s3_check_hash(manifest_path, states["manifest"].get("csv_sha256"))
    metadata_path = dirs1["metadata"]
    catalog_path = metadata_path.parent / "audit_export_report.json"
    if catalog_path.is_file():
        catalog = io_s3_read_json(catalog_path)
        records = [record for record in catalog.get("artifacts", []) if record.get("artifact") == "metadata_clean"]
        if len(records) != 1 or records[0].get("filename") != metadata_path.name:
            raise ValueError("The Sector 1 catalog does not uniquely identify metadata_clean.parquet.")
        hashes[str(metadata_path)] = io_s3_check_hash(metadata_path, records[0].get("file_sha256"))
    else:
        hashes[str(metadata_path)] = io_s3_sha256(metadata_path)
        catalog_path = None

    alignment_input_paths = {
    "metadata_clean.parquet": metadata_path,
    "video_manifest.csv": manifest_path,
    "physical_image_inventory.csv": (
        dirs2["metadata"].parent / "physical_image_inventory.csv"
      ),
    }

    recorded_input_hashes = alignment.get("input_sha256")
    if not isinstance(recorded_input_hashes, dict):
        raise ValueError(
            "Sector 2 alignment has no input_sha256 mapping. "
            "Rerun Sector 2 before continuing."
        )

    for filename, path in alignment_input_paths.items():
        expected = recorded_input_hashes.get(filename)

        if (
            not isinstance(expected, str)
            or re.fullmatch(r"[0-9a-f]{64}", expected) is None
        ):
            raise ValueError(
                f"Sector 2 has no valid input SHA-256 for {filename}."
            )

        # Reuse hashes already verified by the existing loader.
        actual = hashes.get(str(path))
        if actual is None:
            actual = io_s3_sha256(path)

        if actual != expected:
            raise ValueError(
                f"{filename} differs from the input used by "
                "Sector 2 alignment. Use matching upstream artifacts "
                "or rerun Sector 2."
            )

        hashes[str(path)] = actual

    required = {"alignment": {"frame_alignment_confirmed.parquet", "frame_alignment_excluded.csv"},
                "decode": {"decode_audit.csv"}}
    if any(not names.issubset(states[family].get("output_sha256", {})) for family, names in required.items()):
        raise ValueError("The completed audit states lack hashes for the required Sector 3 inputs.")
    inputs = {"metadata_clean": pd.read_parquet(metadata_path),
              "df_alignment_confirmed": pd.read_parquet(dirs2["results"] / "frame_alignment_confirmed.parquet"),
              "video_manifest": pd.read_csv(manifest_path, dtype={"video_key": "string"}),
              "decode_audit": pd.read_csv(dirs2["results"] / "decode_audit.csv", dtype={"video_key": "string"})}
    excluded = pd.read_csv(dirs2["results"] / "frame_alignment_excluded.csv", dtype="string")
    validate_sector3_input_partition(inputs["metadata_clean"], inputs["df_alignment_confirmed"], excluded,
                                    states, inputs["video_manifest"], inputs["decode_audit"])
    provenance = {"source_commit": notebook_sources["source_commit"],
                  "source_notebooks": {sector: {key: value for key, value in notebook_sources[sector].items()
                                                if key not in {"CONFIG", "GIT_CONFIG"}} for sector in literals},
                  "literal_configs": literals, "runtime_storage_root": runtime_root or source2["storage_root"],
                  "runtime_storage_backend": bootstrap.get("storage_backend", source2.get("storage_backend")),
                  "config_sources": {sector: {"kind": "saved_runtime" if path else "notebook_literal_fallback",
                                              "path": str(path) if path else notebook_sources[sector]["notebook_path"],
                                              "sha256": io_s3_sha256(path) if path else notebook_sources[sector]["notebook_sha256"]}
                                     for sector, path in (("sector1", source1_path), ("sector2", source2_path))},
                  "sector1_catalog_verified": catalog_path is not None,
                  "upstream_alignment_run_id": run_id}
    git_source = merge_sector3_config_values(notebook_sources["sector1"]["GIT_CONFIG"], notebook_sources["sector2"]["GIT_CONFIG"])
    if git_source["branch"] != bootstrap["branch"]:
        raise ValueError("Inherited GIT_CONFIG branch differs from the bootstrap branch.")
    if not bootstrap.get("allow_local_test_remote", False) and (
        git_source["repo_url"].rstrip("/").removesuffix(".git")
        != bootstrap["repo_url"].rstrip("/").removesuffix(".git")
    ):
        raise ValueError("Inherited GIT_CONFIG repository differs from the bootstrap repository.")
    git_source["local_repo_dir"] = str(Path(repo).resolve())
    git_source["allow_local_test_remote"] = bootstrap.get("allow_local_test_remote", False)
    if git_source["allow_local_test_remote"]:
        git_source["repo_url"] = bootstrap["repo_url"]
    config = build_sector3_config(source1, source2, sector3_settings, git_source, run_context, provenance)
    state_paths = {
        "alignment": alignment_path,
        "alignment_attempt": alignment_path.with_name(
            "frame_alignment_attempt_state.json"
        ),
        "manifest": (
            dirs2["manifests"] / "video_manifest_state.json"
        ),
        "decode": dirs2["results"] / "decode_audit_state.json",
    }
        # Detect changes to the files used during loading and validation.
    watched_inputs = [
        *alignment_input_paths.values(),
        dirs2["results"] / "frame_alignment_confirmed.parquet",
        dirs2["results"] / "frame_alignment_excluded.csv",
        dirs2["results"] / "decode_audit.csv",
    ]

    for path in watched_inputs:
        io_s3_check_hash(path, hashes[str(path)])

    # The attempt and final state must still match the initial snapshot.
    state_hashes = {
        name: (
            io_s3_check_hash(path, alignment_state_hashes[name])
            if name in alignment_state_hashes
            else io_s3_sha256(path)
        )
        for name, path in state_paths.items()
    }
    lineage = {"upstream_states": states, "input_sha256": hashes,
               "state_sha256": state_hashes,
               "source_config_provenance": provenance, "excluded_annotation_rows": len(excluded),
               "config_conflict_report": pd.DataFrame.from_records(config["config_conflict_report"],
                        columns=["config_key", "sector1_value", "sector2_value", "resolution"])}
    return config, inputs, lineage


In [21]:
REPO_DIR = io_bootstrap_repository(BOOTSTRAP)
NOTEBOOK_SOURCES = io_load_upstream_notebooks(REPO_DIR, BOOTSTRAP["notebook_paths"])

SOURCE_STORAGE_CONFIG = NOTEBOOK_SOURCES["sector2"]["CONFIG"]
STORAGE_BACKEND = BOOTSTRAP.get("storage_backend") or SOURCE_STORAGE_CONFIG.get("storage_backend")
if STORAGE_BACKEND == "google_drive":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

CONFIG, INPUTS, LINEAGE = io_load_sector3_context(
    REPO_DIR, BOOTSTRAP, SECTOR3_SETTINGS, NOTEBOOK_SOURCES, run_context={},
)
LINEAGE = {
    **LINEAGE,
    "config_conflict_report": sector3_semantic_conflict_report(LINEAGE["config_conflict_report"]),
}

EXECUTION_TIMEZONE = CONFIG.get("timezone", "America/Toronto")
EXECUTION_DATETIME = datetime.now(ZoneInfo(EXECUTION_TIMEZONE))
EXECUTION_CONTEXT = {
    "run_id": EXECUTION_DATETIME.strftime("%Y%m%d_%H%M%S_%f"),
    "created_at": EXECUTION_DATETIME.isoformat(),
    "timezone": EXECUTION_TIMEZONE,
}
CONFIG = {**CONFIG, **EXECUTION_CONTEXT}

print("Execution time:", EXECUTION_DATETIME.isoformat())
print("Original annotations:", len(INPUTS["metadata_clean"]))
print("Confirmed annotations:", len(INPUTS["df_alignment_confirmed"]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Execution time: 2026-10-07T20:36:42.861119-04:00
Original annotations: 47239
Confirmed annotations: 47239


### 4. Pure and data-oriented transformations


In [22]:
S3_SOURCE_IDENTITY = (
    "video_key", "frame_number", "image_key", "filename",
    "finding_class_raw", "finding_class", "finding_class_normalized", "finding_category",
    "x1", "x2", "x3", "x4", "y1", "y2", "y3", "y4",
    "bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax",
)
S3_BBOX_FLAGS = (
    "bbox_annotation_present", "bbox_complete", "bbox_invalid", "has_bbox",
)
S3_BBOX_COORDINATES = ("bbox_xmin", "bbox_ymin", "bbox_xmax", "bbox_ymax")
S3_BBOX_STATUSES = (
    "valid", "missing", "incomplete", "invalid_geometry", "out_of_image_bounds",
)


def s3_require_columns(table, columns, name):
    """Validate a table's explicit schema without changing it."""
    missing = sorted(set(columns) - set(table.columns))
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")
    if not table.columns.is_unique:
        raise ValueError(f"{name} contains duplicate column names.")
    return table


def s3_integer_series(series, name, minimum=0):
    """Reject missing, nonfinite, fractional and boolean integer references."""
    numbers = pd.to_numeric(series, errors="coerce")
    finite = pd.Series(np.isfinite(numbers.astype("float64")), index=series.index)
    boolean_input = series.astype("string").str.lower().isin(["true", "false"])
    invalid = numbers.isna() | ~finite | numbers.mod(1).ne(0) | boolean_input
    if minimum is not None:
        invalid = invalid | numbers.lt(minimum)
    invalid = invalid | numbers.ge(2**63) | numbers.lt(-(2**63))
    if invalid.any():
        examples = series.loc[invalid].head(5).astype("string").tolist()
        raise ValueError(f"{name} must contain finite integers; examples: {examples}")
    return numbers.astype("Int64")


def s3_boolean_series(series, name):
    """Read genuine booleans or explicit boolean text; never use string truthiness."""
    normalized = series.astype("string").str.strip().str.lower()
    if not normalized.isin(["true", "false"]).all():
        raise ValueError(f"{name} must contain nonmissing true/false values.")
    return normalized.eq("true").astype("boolean")


def s3_nonempty(series, name):
    result = series.astype("string").str.strip()
    if result.isna().any() or result.eq("").any():
        raise ValueError(f"{name} contains empty identities or labels.")
    return result


def s3_portable_relpaths(series, source_storage_root, name):
    """Lexically convert recorded POSIX paths, with no filesystem access."""
    root = str(source_storage_root).rstrip("/")
    if not root.startswith("/") or re.search(r"(?:^|/)\.\.?(?:/|$)", root):
        raise ValueError("source_storage_root must be an absolute canonical POSIX path.")
    paths = s3_nonempty(series, name)
    unsafe = paths.str.contains(r"(?:^|/)\.\.?(?:/|$)", regex=True) | paths.str.contains("\\", regex=False)
    outside = ~paths.str.startswith(root + "/")
    if (unsafe | outside).any():
        examples = paths.loc[unsafe | outside].head(5).tolist()
        raise ValueError(f"{name} contains paths outside source_storage_root: {examples}")
    return paths.str.slice(len(root) + 1).astype("string")


def s3_annotation_identities(table, identity_columns):
    """Stable content identities preserve every source annotation, including boxes."""
    identity = table.loc[:, identity_columns].copy()
    numeric_columns = [
        column for column in identity_columns
        if column == "frame_number" or column.startswith("bbox_")
        or re.fullmatch(r"[xy][1-4]", column)
    ]
    # A small schema loop normalizes columns; there is no row-wise transformation.
    for column in numeric_columns:
        numeric = pd.to_numeric(identity[column], errors="coerce").astype("Float64")
        original_text = identity[column].astype("string").str.strip()
        identity[column] = numeric.astype("string").where(numeric.notna() | original_text.isna(), "raw:" + original_text)
    canonical = identity.astype("string").fillna("<NULL>")
    hashes = pd.util.hash_pandas_object(canonical, index=False, hash_key="Sector3Identity!")
    encoded = hashes.map(lambda value: f"{value:016x}").astype("string")
    occurrences = encoded.groupby(encoded, sort=False).cumcount().astype("string")
    return "annotation_" + encoded + "_" + occurrences


def s3_frame_ids(video_keys, indices):
    """Unambiguous frame identities remain stable across storage-root changes."""
    keys = video_keys.astype("string")
    return "frame_" + keys.str.len().astype("string") + ":" + keys + ":" + indices.astype("string")


def build_clinical_taxonomy_report(metadata_clean, confirmed, config):
    """Check the declared map against both source and retained clinical labels."""
    mapping = config.get("clinical_group_map")
    if not isinstance(mapping, dict) or not mapping:
        raise ValueError("clinical_group_map must be inherited from the upstream configuration.")
    expected = pd.Series(mapping, name="expected_clinical_group", dtype="string")
    if expected.index.has_duplicates or expected.isna().any() or expected.str.strip().eq("").any():
        raise ValueError("The declared clinical taxonomy contains empty or duplicate mappings.")
    for table, name in ((metadata_clean, "Sector 1 metadata"), (confirmed, "Sector 2 confirmed annotations")):
        s3_require_columns(table, ["finding_class", "finding_class_normalized", "clinical_group"], name)
        labels = s3_nonempty(table["finding_class_normalized"], name + " normalized classes")
        groups = s3_nonempty(table["clinical_group"], name + " clinical groups")
        declared = labels.map(expected)
        wrong = declared.isna() | groups.ne(declared)
        if wrong.any():
            examples = table.loc[wrong, ["finding_class_normalized", "clinical_group"]].assign(
                expected_clinical_group=declared.loc[wrong],
            ).drop_duplicates().head(10)
            raise ValueError(f"{name} disagrees with clinical_group_map: {examples.to_dict('records')}")
    taxonomy = (
        metadata_clean.groupby(["finding_class", "finding_class_normalized", "clinical_group"], dropna=False, observed=True)
        .size().rename("original_annotation_count").reset_index()
    )
    scaffold = expected.rename_axis("finding_class_normalized").reset_index()
    report = scaffold.merge(taxonomy, on="finding_class_normalized", how="left", validate="one_to_many")
    retained_counts = confirmed.groupby("finding_class_normalized", observed=True).size()
    return (
        report.assign(
            clinical_group=report["clinical_group"].fillna(report["expected_clinical_group"]),
            original_annotation_count=report["original_annotation_count"].fillna(0).astype("int64"),
            retained_class_annotation_count=report["finding_class_normalized"].map(retained_counts).fillna(0).astype("int64"),
            mapping_validation_passed=True,
        ).sort_values(["clinical_group", "finding_class_normalized", "finding_class"], kind="stable")
        .reset_index(drop=True)
    )


def build_video_frame_manifest(video_manifest, decode_audit):
    """Use completed decode counts as bounds and preserve container reports separately."""
    s3_require_columns(video_manifest, ["video_key", "width", "height"], "video_manifest")
    s3_require_columns(decode_audit, ["video_key", "frames_read_before_stop", "decoding_completed", "stop_reason"], "decode_audit")
    for table, name in ((video_manifest, "video_manifest"), (decode_audit, "decode_audit")):
        s3_nonempty(table["video_key"], name + ".video_key")
        if not table["video_key"].is_unique:
            raise ValueError(f"{name} contains duplicate video keys.")
    if set(video_manifest["video_key"]) != set(decode_audit["video_key"]):
        raise ValueError("Decode audit must cover exactly the physical video inventory.")
    completed = s3_boolean_series(decode_audit["decoding_completed"], "decoding_completed")
    if not (completed & decode_audit["stop_reason"].eq("read_completed")).all():
        raise ValueError("Sector 3 requires complete successful decode audits for every source video.")
    counts = s3_integer_series(decode_audit["frames_read_before_stop"], "decoded frame counts", minimum=1)
    base = video_manifest.copy()
    reported_column = next((name for name in ("container_reported_frame_count", "container_reported_frames", "frame_count") if name in base), None)
    base = base.assign(container_reported_frame_count=(base[reported_column] if reported_column else pd.NA))
    base = base.drop(columns=["frame_count", "frames_read_before_stop", "decoding_completed", "stop_reason"], errors="ignore")
    audit_columns = [name for name in ("video_key", "frames_read_before_stop", "decoding_completed", "stop_reason", "actual_backend", "backend", "decode_backend", "decoder_backend") if name in decode_audit]
    result = base.merge(decode_audit.loc[:, audit_columns].assign(frames_read_before_stop=counts), on="video_key", how="left", validate="one_to_one")
    return result.assign(
        width=s3_integer_series(result["width"], "video width", minimum=1),
        height=s3_integer_series(result["height"], "video height", minimum=1),
        frame_count=result["frames_read_before_stop"].astype("Int64"),
        frame_count_source="completed_sector2_decode_audit",
    ).sort_values("video_key", kind="stable").reset_index(drop=True)


def build_labelled_frame_manifest(metadata_clean, confirmed, config):
    """One row is one source annotation associated with a confirmed decoded frame."""
    required = [
        "filename", "image_key", "video_key", "frame_number", "finding_class",
        "finding_class_normalized", "clinical_group", "aligned_frame_index",
        "alignment_image_path", "alignment_video_path", "alignment_training_eligible",
        "alignment_mapping_id", "alignment_offset", "alignment_ssim",
        *S3_BBOX_FLAGS, *S3_BBOX_COORDINATES,
    ]
    s3_require_columns(confirmed, required, "df_alignment_confirmed")
    if confirmed.empty:
        raise ValueError("Sector 2 produced no accepted annotations; Sector 3 cannot build labelled evidence.")
    eligible = s3_boolean_series(confirmed["alignment_training_eligible"], "alignment_training_eligible")
    if not eligible.all():
        raise ValueError("The confirmed subset contains annotations that are not alignment eligible.")
    identity_columns = [name for name in S3_SOURCE_IDENTITY if name in metadata_clean]
    s3_require_columns(confirmed, identity_columns, "df_alignment_confirmed source identity")
    source_ids = s3_annotation_identities(metadata_clean, identity_columns)
    accepted_ids = s3_annotation_identities(confirmed, identity_columns)
    if not accepted_ids.isin(source_ids).all():
        raise ValueError("Accepted annotations are not an identity-preserving subset of Sector 1 metadata.")
    source_frame_counts = metadata_clean.groupby(["video_key", "frame_number"], observed=True).size()
    retained_frame_counts = confirmed.groupby(["video_key", "frame_number"], observed=True).size()
    if not retained_frame_counts.eq(source_frame_counts.reindex(retained_frame_counts.index)).all():
        raise ValueError("Confirmed source frames must retain all their source annotations and bounding boxes.")
    indices = s3_integer_series(confirmed["aligned_frame_index"], "aligned_frame_index")
    source_frames = s3_integer_series(confirmed["frame_number"], "source frame_number")
    offsets = s3_integer_series(confirmed["alignment_offset"], "alignment_offset", minimum=None)
    mapping_ids = s3_integer_series(confirmed["alignment_mapping_id"], "alignment_mapping_id")
    scores = pd.to_numeric(confirmed["alignment_ssim"], errors="coerce")
    score_valid = scores.notna() & np.isfinite(scores.astype("float64")) & scores.between(-1, 1)
    if config.get("frame_alignment_min_ssim") is not None:
        score_valid = score_valid & scores.ge(float(config["frame_alignment_min_ssim"]))
    if not score_valid.all():
        raise ValueError("Accepted alignment scores are invalid or below the inherited SSIM threshold.")
    if not indices.eq(source_frames + offsets).all():
        raise ValueError("Sector 2 aligned indices disagree with the preserved per-row source frame and offset.")
    if offsets.groupby(confirmed["video_key"], observed=True).nunique().gt(1).any():
        raise ValueError("Accepted annotations violate Sector 2's consistent-offset rule within a video.")
    source_root = config.get("source_storage_root", config["storage_root"])
    runtime_root = str(config["storage_root"]).rstrip("/")
    if not runtime_root.startswith("/") or re.search(r"(?:^|/)\.\.?(?:/|$)", runtime_root) or "\\" in runtime_root:
        raise ValueError("storage_root must be an absolute canonical POSIX path.")
    image_relative = s3_portable_relpaths(confirmed["alignment_image_path"], source_root, "alignment_image_path")
    video_relative = s3_portable_relpaths(confirmed["alignment_video_path"], source_root, "alignment_video_path")
    result = confirmed.copy().assign(
        annotation_id=accepted_ids, frame_id=s3_frame_ids(confirmed["video_key"], indices),
        frame_number=source_frames, aligned_frame_index=indices,
        opencv_frame_index=indices, frame_index_offset=offsets,
        image_relpath=image_relative, video_relpath=video_relative,
        image_path=runtime_root + "/" + image_relative,
        video_path=runtime_root + "/" + video_relative,
        annotation_status="source_expert_annotation_alignment_confirmed",
        label_source="kvasir_capsule_expert_annotation",
        alignment_training_eligible=eligible, alignment_mapping_id=mapping_ids, alignment_ssim=scores,
    )
    if not result["annotation_id"].is_unique:
        raise RuntimeError("Stable annotation identities are not unique.")
    conflicting_mappings = result.groupby(["video_key", "frame_number"], observed=True)["aligned_frame_index"].nunique()
    if conflicting_mappings.gt(1).any():
        raise ValueError("Annotations of the same source frame have conflicting decoded-frame identities.")
    return result.sort_values(["video_key", "aligned_frame_index", "annotation_id"], kind="stable").reset_index(drop=True)


def build_labelled_frame_bounds_report(labelled_manifest, video_frame_manifest):
    """Count unique decoded frames and stop when an accepted reference is out of bounds."""
    frames = labelled_manifest[["video_key", "aligned_frame_index"]].drop_duplicates()
    checked = frames.merge(video_frame_manifest[["video_key", "frame_count"]], on="video_key", how="left", validate="many_to_one")
    if checked["frame_count"].isna().any():
        raise ValueError("Accepted frames reference a video missing from the decoded inventory.")
    checked = checked.assign(frame_out_of_bounds=checked["aligned_frame_index"].lt(0) | checked["aligned_frame_index"].ge(checked["frame_count"]))
    report = checked.groupby("video_key", as_index=False, observed=True).agg(
        labelled_frame_count=("aligned_frame_index", "size"),
        minimum_aligned_frame_index=("aligned_frame_index", "min"),
        maximum_aligned_frame_index=("aligned_frame_index", "max"),
        video_frame_count=("frame_count", "first"),
        out_of_bounds_frame_count=("frame_out_of_bounds", "sum"),
    )
    report = report.assign(validation_passed=report["out_of_bounds_frame_count"].eq(0))
    if not report["validation_passed"].all():
        failures = report.loc[~report["validation_passed"]].to_dict("records")
        raise ValueError(f"Accepted frame indices are outside decoded video bounds: {failures}")
    return report


def build_frame_inventory_report(metadata_clean, labelled_manifest, video_frame_manifest, config):
    """Published-reference differences are diagnostic, not acceptance requirements."""
    original_frames = metadata_clean[["video_key", "frame_number"]].drop_duplicates()
    accepted_source_frames = labelled_manifest[["video_key", "frame_number"]].drop_duplicates()
    accepted_decoded = labelled_manifest[["video_key", "aligned_frame_index"]].drop_duplicates()
    total = int(video_frame_manifest["frame_count"].sum())
    residual = total - len(accepted_decoded)
    if residual < 0:
        raise RuntimeError("Accepted unique decoded frames exceed the physical video inventory.")
    report = pd.DataFrame({
        "frame_category": [
            "total_decoded_frames", "original_annotation_rows", "original_unique_source_frame_references",
            "accepted_annotation_rows", "accepted_unique_decoded_frames",
            "source_frame_references_without_accepted_evidence", "decoded_frames_without_accepted_evidence",
        ],
        "actual_frame_count": [
            total, len(metadata_clean), len(original_frames), len(labelled_manifest), len(accepted_decoded),
            len(original_frames) - len(accepted_source_frames), residual,
        ],
        "expected_frame_count": [
            config.get("expected_total_extractable_frames", config.get("decode_audit_published_reference_frames", {}).get("all_videos")), None, config.get("expected_labelled_frames"),
            None, None, None, None,
        ],
        "count_unit": ["decoded_frame", "annotation", "source_frame_reference", "annotation", "decoded_frame", "source_frame_reference", "decoded_frame"],
    })
    expected = pd.to_numeric(report["expected_frame_count"], errors="coerce").astype("Int64")
    differences = report["actual_frame_count"].astype("Int64") - expected
    return report.assign(
        expected_frame_count=expected, difference_from_reference=differences,
        reference_matches=differences.eq(0).astype("boolean"),
        validation_passed=True,
        reference_is_diagnostic=True,
    )


def add_bbox_validation_metadata(labelled_manifest, video_frame_manifest):
    """Retain defective boxes, while independently recomputing spatial usability."""
    dimensions = video_frame_manifest[["video_key", "width", "height"]].rename(columns={"width": "source_frame_width", "height": "source_frame_height"})
    base = labelled_manifest.drop(columns=["source_frame_width", "source_frame_height"], errors="ignore")
    result = base.merge(dimensions, on="video_key", how="left", validate="many_to_one", sort=False)
    if result[["source_frame_width", "source_frame_height"]].isna().any(axis=None):
        raise ValueError("Source dimensions are unavailable for one or more accepted annotations.")
    flags = {name: s3_boolean_series(result[name], name) for name in S3_BBOX_FLAGS}
    present, complete = flags["bbox_annotation_present"], flags["bbox_complete"]
    if (complete & ~present).any() or (flags["has_bbox"] & ~complete).any():
        raise ValueError("Bounding-box completeness flags are internally inconsistent.")
    coords = result.loc[:, S3_BBOX_COORDINATES].apply(pd.to_numeric, errors="coerce")
    finite = pd.Series(np.isfinite(coords.astype("float64")).all(axis=1), index=result.index)
    geometry = complete & finite & coords["bbox_xmax"].gt(coords["bbox_xmin"]) & coords["bbox_ymax"].gt(coords["bbox_ymin"])
    in_bounds = (
        geometry & coords["bbox_xmin"].ge(0) & coords["bbox_ymin"].ge(0)
        & coords["bbox_xmax"].le(result["source_frame_width"])
        & coords["bbox_ymax"].le(result["source_frame_height"])
    )
    states = np.select(
        [~present.to_numpy(dtype=bool), (present & ~complete).to_numpy(dtype=bool),
         (complete & ~geometry).to_numpy(dtype=bool), (geometry & ~in_bounds).to_numpy(dtype=bool)],
        ["missing", "incomplete", "invalid_geometry", "out_of_image_bounds"], default="valid",
    )
    return result.assign(
        bbox_annotation_present=present, bbox_complete=complete,
        bbox_invalid=(present & ~geometry).astype("boolean"), has_bbox=geometry.astype("boolean"),
        bbox_within_image_bounds=in_bounds.astype("boolean"), bbox_spatially_usable=in_bounds.astype("boolean"),
        bbox_validation_status=pd.Series(states, index=result.index, dtype="string"),
        bbox_bounds_policy="normalized_xyxy_max_edge_may_equal_source_width_or_height",
    )


def validate_bbox_validation_report(report, labelled_manifest, config):
    """Validate identity coverage and status/flag coherence rather than lengths alone."""
    s3_require_columns(report, ["annotation_id", "bbox_validation_status", "bbox_spatially_usable"], "bbox_validation_report")
    if not report["annotation_id"].is_unique or set(report["annotation_id"]) != set(labelled_manifest["annotation_id"]):
        raise RuntimeError("BBox report does not cover every retained annotation exactly once.")
    if not report["bbox_validation_status"].isin(S3_BBOX_STATUSES).all():
        raise RuntimeError("BBox report contains missing or unexpected validation states.")
    usable = s3_boolean_series(report["bbox_spatially_usable"], "bbox_spatially_usable")
    if not usable.eq(report["bbox_validation_status"].eq("valid")).all():
        raise RuntimeError("BBox report validity states disagree with spatial-usability flags.")
    settings = config.get("sector3", {})
    if settings.get("require_any_bbox", True) and report["bbox_validation_status"].eq("missing").all():
        raise RuntimeError("No bbox annotations were detected; check the inherited metadata schema.")
    if settings.get("require_valid_bbox", True) and not usable.any():
        raise RuntimeError("No bbox is spatially usable; check coordinate units and source dimensions.")
    return report


def build_class_coverage_tables(metadata_clean, labelled_manifest, video_frame_manifest, config):
    """Preserve multi-label counts, retain zero-support classes, and expose rare support."""
    classes = pd.Index(sorted(config["clinical_group_map"]), name="finding_class_normalized")
    source_videos = pd.Index(sorted(metadata_clean["video_key"].unique()), name="video_key")
    accepted_pairs = labelled_manifest[["video_key", "aligned_frame_index", "finding_class_normalized"]].drop_duplicates()
    counts = pd.crosstab(accepted_pairs["video_key"], accepted_pairs["finding_class_normalized"])
    counts = counts.reindex(index=source_videos, columns=classes, fill_value=0).astype("int64")
    binary = counts.gt(0).astype("int8")
    source_unique = metadata_clean[["video_key", "frame_number", "finding_class_normalized"]].drop_duplicates()
    frame_support = accepted_pairs.groupby("finding_class_normalized", observed=True).size().reindex(classes, fill_value=0)
    annotation_support = labelled_manifest.groupby("finding_class_normalized", observed=True).size().reindex(classes, fill_value=0)
    original_support = metadata_clean.groupby("finding_class_normalized", observed=True).size().reindex(classes, fill_value=0)
    video_support = binary.sum(axis=0).astype("int64")
    distribution = pd.DataFrame({
        "finding_class_normalized": classes,
        "clinical_group": pd.Series(classes).map(config["clinical_group_map"]).to_numpy(),
        "original_annotation_count": original_support.to_numpy(),
        "original_unique_source_frame_count": source_unique.groupby("finding_class_normalized", observed=True).size().reindex(classes, fill_value=0).to_numpy(),
        "retained_annotation_count": annotation_support.to_numpy(),
        "retained_unique_decoded_frame_count": frame_support.to_numpy(),
        "retained_video_count": video_support.to_numpy(),
    })
    distribution = distribution.assign(
        excluded_annotation_count=distribution["original_annotation_count"] - distribution["retained_annotation_count"],
        annotation_retention_percent=(100 * distribution["retained_annotation_count"] / distribution["original_annotation_count"].replace(0, np.nan)),
        class_missing_after_alignment=distribution["retained_annotation_count"].eq(0),
        rare_video_support=distribution["retained_video_count"].lt(int(config.get("sector3", {}).get("rare_video_support_threshold", 3))),
    )
    videos = pd.DataFrame({"video_key": video_frame_manifest["video_key"]})
    original_per_video = metadata_clean.groupby("video_key", observed=True).size()
    accepted_per_video = labelled_manifest.groupby("video_key", observed=True).size()
    unique_per_video = accepted_pairs[["video_key", "aligned_frame_index"]].drop_duplicates().groupby("video_key", observed=True).size()
    support = videos.assign(
        original_annotation_count=videos["video_key"].map(original_per_video).fillna(0).astype("int64"),
        retained_annotation_count=videos["video_key"].map(accepted_per_video).fillna(0).astype("int64"),
        retained_unique_frame_count=videos["video_key"].map(unique_per_video).fillna(0).astype("int64"),
        retained_class_count=videos["video_key"].map(binary.sum(axis=1)).fillna(0).astype("int64"),
    )
    support = support.assign(
        source_has_annotations=support["original_annotation_count"].gt(0),
        has_accepted_labelled_evidence=support["retained_annotation_count"].gt(0),
    )
    return {
        "video_class_matrix": binary.rename_axis(columns=None).reset_index(),
        "video_class_counts": counts.rename_axis(columns=None).reset_index(),
        "class_distribution_report": distribution,
        "video_class_support": support,
    }


def build_sector3_tables(metadata_clean, df_alignment_confirmed, video_manifest, decode_audit, config):
    """Build all Sector 3 tables from explicit inputs, without I/O or input mutation."""
    s3_require_columns(metadata_clean, ["video_key", "frame_number", "image_key"], "metadata_clean")
    s3_nonempty(metadata_clean["video_key"], "source video_key")
    s3_integer_series(metadata_clean["frame_number"], "source frame_number")
    taxonomy = build_clinical_taxonomy_report(metadata_clean, df_alignment_confirmed, config)
    videos = build_video_frame_manifest(video_manifest, decode_audit)
    if not metadata_clean["video_key"].isin(videos["video_key"]).all():
        raise ValueError("Source annotations reference videos outside the physical video inventory.")
    manifest = build_labelled_frame_manifest(metadata_clean, df_alignment_confirmed, config)
    bounds = build_labelled_frame_bounds_report(manifest, videos)
    manifest = add_bbox_validation_metadata(manifest, videos)
    bbox_columns = [
        "annotation_id", "frame_id", "video_key", "frame_number", "aligned_frame_index", "filename",
        "image_relpath", "finding_class_normalized", *S3_BBOX_FLAGS, *S3_BBOX_COORDINATES,
        "source_frame_width", "source_frame_height", "bbox_within_image_bounds",
        "bbox_spatially_usable", "bbox_validation_status",
    ]
    bbox = validate_bbox_validation_report(manifest[bbox_columns].copy(), manifest, config)
    summary = bbox["bbox_validation_status"].value_counts().reindex(S3_BBOX_STATUSES, fill_value=0).rename("annotation_count").rename_axis("bbox_validation_status").reset_index()
    return {
        "labelled_frame_manifest": manifest,
        "video_frame_manifest": videos,
        "clinical_taxonomy_report": taxonomy,
        "labelled_frame_bounds_report": bounds,
        "frame_inventory_report": build_frame_inventory_report(metadata_clean, manifest, videos, config),
        "bbox_validation_report": bbox,
        "bbox_validation_summary": summary,
        **build_class_coverage_tables(metadata_clean, manifest, videos, config),
        "config_conflict_report": pd.DataFrame(columns=["config_key", "sector1_value", "sector2_value", "resolution"]),
    }

def _s3_core_code_value(code):
    """Describe executable code without filenames, source lines or debug positions."""
    fields = (
        "co_argcount", "co_posonlyargcount", "co_kwonlyargcount", "co_nlocals",
        "co_stacksize", "co_flags", "co_code", "co_consts", "co_names", "co_varnames",
        "co_name", "co_qualname", "co_cellvars", "co_freevars", "co_exceptiontable",
    )
    return {field: _s3_core_value(getattr(code, field)) for field in fields if hasattr(code, field)}


def _s3_core_value(value, active=None):
    """Canonicalize scalar/container code values and defaults without changing them."""
    if isinstance(value, CodeType):
        return {"code": _s3_core_code_value(value)}
    if value is Ellipsis:
        return {"ellipsis": True}
    if value is None or isinstance(value, (str, bool, int)):
        return value
    if isinstance(value, float):
        return {"float": value.hex()} if math.isfinite(value) else {"float": repr(value)}
    if isinstance(value, complex):
        return {"complex": [_s3_core_value(value.real), _s3_core_value(value.imag)]}
    if isinstance(value, bytes):
        return {"bytes": value.hex()}
    if not isinstance(value, (dict, list, tuple, set, frozenset)):
        raise TypeError(f"Unsupported active core identity value: {type(value).__name__}")
    active = set() if active is None else active
    identity = id(value)
    if identity in active:
        raise ValueError("Cyclic containers cannot define a deterministic core identity.")
    nested = active | {identity}
    if isinstance(value, dict):
        pairs = [[_s3_core_value(key, nested), _s3_core_value(item, nested)] for key, item in value.items()]
        return {"dict": sorted(pairs, key=lambda pair: _s3_core_identity_json(pair[0]))}
    items = [_s3_core_value(item, nested) for item in value]
    if isinstance(value, (set, frozenset)):
        items = sorted(items, key=_s3_core_identity_json)
    return {type(value).__name__: items}


def _s3_core_identity_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)


def _s3_core_referenced_names(code):
    """Include globals referenced inside comprehensions and other nested code."""
    children = (item for item in code.co_consts if isinstance(item, CodeType))
    return set(code.co_names).union(*( _s3_core_referenced_names(child) for child in children))


def _s3_core_function_value(function):
    closure = tuple(cell.cell_contents for cell in (function.__closure__ or ()))
    return {
        "code": _s3_core_code_value(function.__code__),
        "defaults": _s3_core_value(function.__defaults__),
        "kwdefaults": _s3_core_value(function.__kwdefaults__),
        "closure": _s3_core_value(closure),
    }


def sector3_runtime_core_sha256(entrypoint, namespace, dependency_versions):
    """Fingerprint active core functions, referenced S3 constants and library versions.

    Only Python functions reachable through executable global references and
    referenced ``S3_*`` constants participate. Unrelated configuration, clocks and
    notebook state are ignored. Recursive function graphs terminate through a
    visited-name set; repeated cell executions keep the same identity.
    """
    if not isinstance(entrypoint, FunctionType) or not isinstance(namespace, dict):
        raise TypeError("Provide a Python entrypoint function and its explicit namespace dictionary.")
    if not isinstance(dependency_versions, dict):
        raise TypeError("Dependency versions must be an explicit dictionary.")
    pending = {"entrypoint": entrypoint}
    functions, constants = {}, {}
    while pending:
        name = min(pending)
        function = pending.pop(name)
        if name in functions:
            continue
        functions[name] = _s3_core_function_value(function)
        for reference in sorted(_s3_core_referenced_names(function.__code__)):
            value = namespace.get(reference)
            if isinstance(value, FunctionType) and reference not in functions:
                pending[reference] = value
            elif reference.startswith("S3_") and reference in namespace:
                constants[reference] = _s3_core_value(value)
    payload = {
        "schema_version": 1, "functions": functions, "constants": constants,
        "dependency_versions": _s3_core_value(dependency_versions),
    }
    return hashlib.sha256(_s3_core_identity_json(payload).encode("utf-8")).hexdigest()


### 5. Reuse verified tables or compute changed transformations

In [23]:
P3_STATE_PATH = "configs/sector3_state.json"
P3_RECEIPT_PATH = "configs/sector3_publication_receipt.json"
P3_NULL = "\\N"


def _p3_json_default(value):
    if value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, pd.DataFrame):
        return {"type": "dataframe", "columns": list(value.columns),
                "dtypes": {str(column): str(dtype) for column, dtype in value.dtypes.items()},
                "records": value.astype(object).where(value.notna(), None).to_dict("records")}
    if isinstance(value, (Path, PurePosixPath, datetime, pd.Timestamp)):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (set, tuple)):
        return sorted(value) if isinstance(value, set) else list(value)
    raise TypeError(f"Unsupported Sector 3 JSON value: {type(value).__name__}")


def _p3_json_text(value):
    return json.dumps(value, sort_keys=True, indent=2, ensure_ascii=False,
                      allow_nan=False, default=_p3_json_default) + "\n"


def _p3_json_hash(value):
    return hashlib.sha256(_p3_json_text(value).encode("utf-8")).hexdigest()


def sector3_lineage_fingerprint(lineage):
    """Identify upstream evidence independently of the current checkout HEAD."""
    if not isinstance(lineage, dict):
        raise TypeError("Sector 3 lineage must be a dictionary.")

    identity = copy.deepcopy(lineage)

    source = identity.get("source_config_provenance")
    if isinstance(source, dict):
        source.pop("source_commit", None)

    return _p3_json_hash(identity)


def _p3_file_hash(path):
    with Path(path).open("rb") as source:
        return hashlib.file_digest(source, "sha256").hexdigest()


def _p3_relative(value):
    path = PurePosixPath(str(value))
    if path.is_absolute() or ".." in path.parts or len(path.parts) != 2:
        raise ValueError(f"Sector 3 artifacts must be flat relative paths: {value!r}")
    if path.parts[0] not in {"configs", "results", "reports"}:
        raise ValueError(f"Unexpected Sector 3 artifact group: {value!r}")
    return path.as_posix()


def _p3_root(config):
    path = Path(config["sector3"]["local_output_dir"])
    return (path if path.is_absolute() else Path(config["storage_root"]) / path).resolve()


def _p3_safe_path(root, relative):
    path = root / _p3_relative(relative)
    if path.is_symlink() or root not in path.resolve().parents or path.parent.is_symlink():
        raise RuntimeError(f"Unsafe Sector 3 artifact path: {relative}")
    return path


def _p3_write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(_p3_json_text(value), encoding="utf-8")
    if _p3_json_text(json.loads(path.read_text(encoding="utf-8"))) != _p3_json_text(value):
        raise RuntimeError(f"JSON readback failed: {path.name}")


def _p3_schema(frame):
    columns = []
    for column, dtype in frame.dtypes.items():
        descriptor = {"name": column, "dtype": str(dtype)}
        if isinstance(dtype, pd.StringDtype):
            descriptor["string_storage"] = dtype.storage
        if str(dtype) == "object":
            descriptor["object_kind"] = pd.api.types.infer_dtype(frame[column], skipna=True)
        if isinstance(dtype, pd.CategoricalDtype):
            descriptor.update({"categories": list(dtype.categories), "ordered": dtype.ordered})
        columns.append(descriptor)
    return {"rows": len(frame), "columns": columns, "column_axis_name": frame.columns.name,
            "csv_null_marker": P3_NULL, "schema_version": 1}


def sector3_table_fingerprint(frame):
    """Hash every value and its dtype schema; no clock or execution metadata is added."""
    if frame.columns.duplicated().any() or not all(isinstance(column, str) for column in frame.columns):
        raise ValueError("Sector 3 tables require unique string column names.")
    normalized = frame.reset_index(drop=True)
    digest = hashlib.sha256(_p3_json_text(_p3_schema(normalized)).encode("utf-8"))
    try:
        # Missing values have one semantic identity, including None/NaN/pd.NA in object columns.
        hash_values = normalized.astype(object).where(normalized.notna(), None)
        values = pd.util.hash_pandas_object(hash_values, index=False, categorize=False)
    except TypeError as error:
        raise TypeError("Sector 3 tables must contain scalar values; canonicalize nested conflict values first.") from error
    digest.update(values.to_numpy(dtype="uint64").astype("<u8", copy=False).tobytes())
    return digest.hexdigest()


def _p3_restore_csv(path, schema):
    frame = pd.read_csv(path, dtype="string", keep_default_na=False)
    expected = [descriptor["name"] for descriptor in schema["columns"]]
    if list(frame.columns) != expected or len(frame) != schema["rows"]:
        raise RuntimeError(f"CSV dimensions differ from saved schema: {path.name}")
    for descriptor in schema["columns"]:
        name, dtype = descriptor["name"], descriptor["dtype"]
        values = frame[name]
        nulls = values.eq(P3_NULL)
        values = values.mask(nulls, pd.NA)
        if dtype == "object":
            kind = descriptor.get("object_kind", "string")
            if kind in {"integer", "floating", "mixed-integer-float", "decimal"}:
                values = pd.to_numeric(values, errors="raise").astype(object)
            elif kind == "boolean":
                values = values.map({"True": True, "False": False}).astype(object)
            elif kind not in {"string", "unicode", "bytes", "empty"}:
                raise RuntimeError(f"Unsupported object CSV dtype {kind!r}: canonicalize this table before saving.")
            else:
                values = values.astype(object)
            frame[name] = values.mask(nulls, None)
        elif dtype in {"bool", "boolean"}:
            if (~nulls & ~values.isin(["True", "False"])).any():
                raise RuntimeError(f"Invalid saved boolean values: {path.name}/{name}")
            frame[name] = values.map({"True": True, "False": False}).astype(dtype)
        elif dtype == "string":
            frame[name] = values.astype(pd.StringDtype(storage=descriptor.get("string_storage", "python")))
        elif dtype == "category":
            category = pd.CategoricalDtype(descriptor["categories"], ordered=descriptor["ordered"])
            if pd.api.types.infer_dtype(descriptor["categories"], skipna=True) in {"integer", "floating", "mixed-integer-float"}:
                values = pd.to_numeric(values, errors="raise")
            frame[name] = values.astype(category)
        elif dtype.startswith("timedelta64"):
            frame[name] = pd.to_timedelta(values).astype(dtype)
        else:
            frame[name] = values.astype(dtype)
    frame.columns.name = schema.get("column_axis_name")
    return frame


def _p3_read_table(root, descriptor):
    path = _p3_safe_path(root, descriptor["relative_path"])
    schema_path = _p3_safe_path(root, descriptor["schema_path"])
    schema = json.loads(schema_path.read_text(encoding="utf-8"))
    if schema.get("fingerprint") != descriptor["fingerprint"] or schema.get("artifact") != descriptor["relative_path"]:
        raise RuntimeError("Table schema disagrees with the committed catalog.")
    frame = pd.read_parquet(path) if path.suffix == ".parquet" else _p3_restore_csv(path, schema)
    if sector3_table_fingerprint(frame) != descriptor["fingerprint"]:
        raise RuntimeError(f"Typed table readback fingerprint failed: {path.name}")
    return frame


def _p3_catalog_meaning(state):
    return {key: state[key] for key in (
        "schema_version", "complete", "processing_signature", "semantic_config_fingerprint",
        "input_sha256", "core_source_sha256", "config_snapshot", "tables", "summary", "artifacts", "obsolete_paths")}


def _p3_processing_provenance(lineage):
    selected = {}
    for source, digest in lineage.get("input_sha256", {}).items():
        filename = PurePosixPath(str(source)).name
        if filename not in S3_CONSUMED_INPUT_FILENAMES:
            continue
        if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):
            raise ValueError(f"Invalid consumed-input SHA-256 in lineage: {filename}")
        if filename in selected and selected[filename] != digest:
            raise ValueError(f"Conflicting consumed-input hashes in lineage: {filename}")
        selected[filename] = digest
    core_hash = lineage.get("core_source_sha256")
    if core_hash is not None and (not isinstance(core_hash, str) or not re.fullmatch(r"[0-9a-f]{64}", core_hash)):
        raise ValueError("Invalid pure-core SHA-256 in lineage.")
    return {"input_sha256": selected, "core_source_sha256": core_hash}


def _p3_read_state(root):
    state_path = _p3_safe_path(root, P3_STATE_PATH)
    if not state_path.exists():
        return None
    try:
        state = json.loads(state_path.read_text(encoding="utf-8"))
        if state.get("schema_version") != 2 or state.get("complete") is not True:
            raise RuntimeError("Sector 3 catalog is incomplete or uses an obsolete layout.")
        if state.get("content_id") != _p3_json_hash(_p3_catalog_meaning(state)):
            raise RuntimeError("Sector 3 catalog content fingerprint is invalid.")
        records = {record["relative_path"]: record for record in state["artifacts"]}
        if len(records) != len(state["artifacts"]) or P3_STATE_PATH in records or P3_RECEIPT_PATH in records:
            raise RuntimeError("Sector 3 catalog contains duplicate or operational artifact entries.")
        for relative, record in records.items():
            path = _p3_safe_path(root, relative)
            if not path.is_file() or path.stat().st_size != record["bytes"] or _p3_file_hash(path) != record["sha256"]:
                raise RuntimeError(f"Sector 3 artifact checksum verification failed: {relative}")
        if state["config_snapshot"] not in records:
            raise RuntimeError("Sector 3 config snapshot is absent from its catalog.")
        config_envelope = json.loads((root / state["config_snapshot"]).read_text(encoding="utf-8"))
        if (config_envelope.get("semantic_config_fingerprint") != state["semantic_config_fingerprint"]
                or sector3_semantic_config_fingerprint(config_envelope["effective_config"]) != state["semantic_config_fingerprint"]):
            raise RuntimeError("Sector 3 configuration snapshot disagrees with its catalog.")
        for field in ("summary_relative_path", "report_relative_path"):
            if state["summary"][field] not in records:
                raise RuntimeError("Sector 3 summary or executive report is absent from its catalog.")
        summary = json.loads((root / state["summary"]["summary_relative_path"]).read_text(encoding="utf-8"))
        if _p3_json_hash(summary) != state["summary"]["fingerprint"]:
            raise RuntimeError("Sector 3 summary disagrees with its catalog.")
        if records[state["summary"]["report_relative_path"]]["sha256"] != state["summary"]["report_fingerprint"]:
            raise RuntimeError("Sector 3 executive report disagrees with its catalog.")
        for descriptor in state["tables"].values():
            if descriptor["relative_path"] not in records or descriptor["schema_path"] not in records:
                raise RuntimeError("Sector 3 table or schema is absent from its catalog.")
            schema = json.loads((root / descriptor["schema_path"]).read_text(encoding="utf-8"))
            if schema.get("artifact") != descriptor["relative_path"] or schema.get("fingerprint") != descriptor["fingerprint"]:
                raise RuntimeError("Sector 3 saved table schema disagrees with the catalog.")
        for obsolete in state["obsolete_paths"]:
            if PurePosixPath(_p3_relative(obsolete)).parts[0] != "results" or obsolete in records:
                raise RuntimeError("Invalid obsolete-result deletion authorization.")
        return state
    except (OSError, ValueError, KeyError, TypeError) as error:
        raise RuntimeError(f"Cannot verify the existing Sector 3 catalog: {error}") from error


def io_load_cached_sector3(config, processing_signature):
    """Read and verify existing tables before invoking any pure transformations."""
    root = _p3_root(config)
    state = _p3_read_state(root)
    if state is None or state["processing_signature"] != processing_signature:
        return None
    tables = {name: _p3_read_table(root, descriptor) for name, descriptor in state["tables"].items()}
    return {"tables": tables, "state": state}


def _p3_summary(tables):
    labelled = tables["labelled_frame_manifest"]
    clean_records = lambda frame: frame.astype(object).where(frame.notna(), None).to_dict("records")
    return {
        "phase": 2, "sector": 3, "annotation_rows": len(labelled),
        "accepted_unique_video_frames": len(labelled[["video_key", "aligned_frame_index"]].drop_duplicates()),
        "labelled_video_count": int(labelled["video_key"].nunique()),
        "spatially_usable_annotation_rows": int(labelled["bbox_spatially_usable"].fillna(False).sum()),
        "table_row_counts": {name: len(frame) for name, frame in sorted(tables.items())},
        "inventory": clean_records(tables["frame_inventory_report"]),
        "bbox_validation": clean_records(tables["bbox_validation_summary"]),
        "units": {"manifest": "source annotation", "frame_identity": ["video_key", "aligned_frame_index"],
                  "video_class_matrix": "binary annotated-class presence"},
    }


def _p3_executive(summary):
    return (
        "# Phase 2 — Sector 3: Clinical Taxonomy and Labelled-Frame Manifest\n\n"
        "This report audits clinical-label mappings, preserves source annotations and alignment provenance, "
        "validates decoded-frame bounds, and flags bounding boxes suitable for spatial grounding. "
        "It includes frame inventories, class distributions, and a binary video/class matrix for downstream stratification.\n\n"
        f"The accepted manifest contains **{summary['annotation_rows']:,} annotation rows**, "
        f"**{summary['accepted_unique_video_frames']:,} distinct video frames**, and "
        f"**{summary['labelled_video_count']:,} labelled videos**. "
        f"**{summary['spatially_usable_annotation_rows']:,} annotation rows** have usable bounding boxes.\n\n"
        "Published frame counts are diagnostic references. Frames without accepted evidence are not "
        "automatically clinically negative or unlabelled in the original dataset. Zero class presence means "
        "no accepted annotation. This sector does not assign splits or run model inference.\n\n"
        "Configuration snapshots and readable report versions are created only after meaningful changes. "
        "Results use stable filenames; identical repeat executions reuse checked saved transformations and files.\n"
    )


def _p3_write_table(stage, name, frame, fingerprint, stamp, config):
    results = name in {"labelled_frame_manifest", "video_frame_manifest", "video_class_matrix", "video_class_counts"}
    base = f"results/{name}" if results else f"reports/{name}_{stamp}"
    relative = base + (".parquet" if name in {"labelled_frame_manifest", "video_frame_manifest"} else ".csv")
    target = stage / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if relative.endswith(".parquet"):
        frame.to_parquet(target, index=False)
        restored = pd.read_parquet(target)
        pd.testing.assert_frame_equal(frame.reset_index(drop=True), restored, check_exact=True, check_dtype=True)
    else:
        serialized = frame.astype("string")
        if serialized.eq(P3_NULL).fillna(False).any().any():
            raise ValueError(f"Reserved CSV null marker occurs in {name}.")
        serialized = serialized.fillna(P3_NULL)
        serialized.to_csv(target, index=False)
        threshold = int(config["sector3"].get("csv_compress_threshold_bytes", 10 * 1024 ** 2))
        if target.stat().st_size > threshold:
            relative += ".gz"
            compressed = stage / relative
            serialized.to_csv(compressed, index=False, compression={"method": "gzip", "mtime": 0})
            target.unlink()
            target = compressed
        schema = _p3_schema(frame)
        restored = _p3_restore_csv(target, schema)
        pd.testing.assert_frame_equal(frame.reset_index(drop=True), restored, check_exact=True, check_dtype=True)
    schema_path = base + "_schema.json"
    schema = {**_p3_schema(frame), "artifact": relative, "fingerprint": fingerprint}
    _p3_write_json(stage / schema_path, schema)
    return {"relative_path": relative, "schema_path": schema_path, "fingerprint": fingerprint}


def _p3_install(stage, root, changed_paths, obsolete_paths):
    """Install already verified files; atomically replace the catalog last and roll back errors."""
    if not root.exists():
        stage.rename(root)
        return
    backup = Path(tempfile.mkdtemp(prefix=".sector3-backup-", dir=root.parent))
    installed, moved = [], []
    try:
        for relative in [*changed_paths, *obsolete_paths]:
            target = _p3_safe_path(root, relative)
            if target.exists():
                saved = backup / relative
                saved.parent.mkdir(parents=True, exist_ok=True)
                target.replace(saved)
                moved.append(relative)
            source = stage / relative
            if source.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                source.replace(target)
                installed.append(relative)
    except BaseException:
        for relative in reversed(installed):
            (root / relative).unlink(missing_ok=True)
        for relative in moved:
            target = root / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            (backup / relative).replace(target)
        raise
    finally:
        shutil.rmtree(backup, ignore_errors=True)


def io_persist_sector3(tables, config, lineage, processing_signature=None):
    """Compare content first; an equivalent repeat creates and touches no output files."""
    if not processing_signature or not isinstance(processing_signature, str):
        raise ValueError("Pass the explicit Sector 3 processing signature before persisting.")
    if not tables or any(not isinstance(frame, pd.DataFrame) for frame in tables.values()):
        raise TypeError("Sector 3 outputs must be a non-empty mapping of DataFrames.")
    if any(not re.fullmatch(r"[a-z][a-z0-9_]*", name) for name in tables):
        raise ValueError("Unsafe Sector 3 table name.")
    root = _p3_root(config)
    previous = _p3_read_state(root)
    config_hash = sector3_semantic_config_fingerprint(config)
    provenance = _p3_processing_provenance(lineage)
    saved_lineage = None

    if previous is not None:
        snapshot_path = _p3_safe_path(
            root,
            previous["config_snapshot"],
        )
        saved_envelope = json.loads(
            snapshot_path.read_text(encoding="utf-8")
        )
        saved_lineage = saved_envelope.get("lineage")

    same_lineage = (
        isinstance(saved_lineage, dict)
        and sector3_lineage_fingerprint(saved_lineage)
        == sector3_lineage_fingerprint(lineage)
    )
    fingerprints = {name: sector3_table_fingerprint(frame) for name, frame in tables.items()}
    summary = _p3_summary(tables)
    summary_hash = _p3_json_hash(summary)
    same_tables = previous is not None and set(fingerprints) == set(previous["tables"]) and all(
        previous["tables"][name]["fingerprint"] == fingerprint for name, fingerprint in fingerprints.items())
    same_provenance = previous is not None and all(previous[key] == value for key, value in provenance.items())
    if (
        previous is not None
        and same_tables
        and same_provenance
        and same_lineage
        and previous["semantic_config_fingerprint"] == config_hash
        and previous["summary"]["fingerprint"] == summary_hash
        and previous["processing_signature"] == processing_signature
    ):
        return {
            "state": previous,
            "local_root": str(root),
            "artifacts": previous["artifacts"],
            "status": "unchanged",
            "changed_paths": [],
        }
    root.parent.mkdir(parents=True, exist_ok=True)
    stage = Path(tempfile.mkdtemp(prefix=".sector3-stage-", dir=root.parent))
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    try:
        descriptors = {}
        catalog = {record["relative_path"]: record for record in previous["artifacts"]} if previous else {}
        obsolete = set(previous.get("obsolete_paths", [])) if previous else set()
        config_snapshot = (
            previous["config_snapshot"]
            if previous is not None
            else None
        )

        if (
            previous is None
            or previous["semantic_config_fingerprint"] != config_hash
            or not same_lineage
        ):
            config_snapshot = f"configs/sector3_config_{stamp}.json"

            envelope = {
                "schema_version": 2,
                "changed_at_utc": stamp,
                "semantic_config_fingerprint": config_hash,
                "effective_config": config,
                "lineage": lineage,
            }

            _p3_write_json(stage / config_snapshot, envelope)

        for name, frame in sorted(tables.items()):
            old = previous["tables"].get(name) if previous else None
            if old and old["fingerprint"] == fingerprints[name]:
                descriptors[name] = old
            else:
                new = _p3_write_table(stage, name, frame, fingerprints[name], stamp, config)
                descriptors[name] = new
                if old and old["relative_path"] != new["relative_path"] and old["relative_path"].startswith("results/"):
                    obsolete.add(old["relative_path"])
                    catalog.pop(old["relative_path"], None)
        if previous:
            for name, old in previous["tables"].items():
                if name not in descriptors and old["relative_path"].startswith("results/"):
                    for relative in (old["relative_path"], old["schema_path"]):
                        obsolete.add(relative)
                        catalog.pop(relative, None)
        previous_summary = previous["summary"] if previous else {}
        summary_path = previous_summary.get("summary_relative_path")
        if previous_summary.get("fingerprint") != summary_hash:
            summary_path = f"reports/summary_{stamp}.json"
            _p3_write_json(stage / summary_path, summary)
        report_text = _p3_executive(summary)
        report_hash = hashlib.sha256(report_text.encode("utf-8")).hexdigest()
        report_path = previous_summary.get("report_relative_path")
        prior_report_hash = catalog.get(report_path, {}).get("sha256")
        if prior_report_hash != report_hash:
            report_path = f"reports/executive_report_{stamp}.md"
            (stage / report_path).write_text(report_text, encoding="utf-8")
        summary_descriptor = {"fingerprint": summary_hash, "summary_relative_path": summary_path,
                              "report_fingerprint": report_hash, "report_relative_path": report_path}
        max_bytes = int(config["github"].get("max_file_size_mib", 20)) * 1024 ** 2
        changed = []
        for path in sorted(stage.rglob("*")):
            if path.is_file():
                relative = path.relative_to(stage).as_posix()
                if path.stat().st_size > max_bytes:
                    raise ValueError(f"Artifact exceeds explicit Git publication size limit: {relative}")
                record = {"relative_path": relative, "sha256": _p3_file_hash(path), "bytes": path.stat().st_size}
                if relative in catalog and catalog[relative] == record:
                    path.unlink()
                else:
                    catalog[relative] = record
                    changed.append(relative)
        obsolete.difference_update(catalog)
        state = {"schema_version": 2, "complete": True, "processing_signature": processing_signature,
                 "semantic_config_fingerprint": config_hash, "config_snapshot": config_snapshot,
                 **provenance,
                 "tables": descriptors, "summary": summary_descriptor,
                 "artifacts": [catalog[path] for path in sorted(catalog)], "obsolete_paths": sorted(obsolete)}
        state["content_id"] = _p3_json_hash(_p3_catalog_meaning(state))
        state["changed_at_utc"] = stamp
        _p3_write_json(stage / P3_STATE_PATH, state)
        changed.append(P3_STATE_PATH)
        # Catalog is last; obsolete files are removed first so it describes a complete visible state.
        install_order = [path for path in changed if path != P3_STATE_PATH] + sorted(obsolete) + [P3_STATE_PATH]
        _p3_install(stage, root, install_order, [])
        return {"state": state, "local_root": str(root), "artifacts": state["artifacts"], "status": "updated", "changed_paths": changed}
    finally:
        shutil.rmtree(stage, ignore_errors=True)


def io_checked_sector3_bundle(persisted, config):
    """Verify the committed stable catalog and return the exact publication whitelist."""
    root = _p3_root(config)
    if str(root) != persisted["local_root"]:
        raise RuntimeError("Persisted Sector 3 root differs from the current configuration.")
    state = _p3_read_state(root)
    if state is None or state != persisted["state"]:
        raise RuntimeError("Supplied Sector 3 state differs from the committed catalog.")
    if state["semantic_config_fingerprint"] != sector3_semantic_config_fingerprint(config):
        raise RuntimeError("Effective authored configuration changed after persistence.")
    records = list(state["artifacts"])
    path = root / P3_STATE_PATH
    records.append({"relative_path": P3_STATE_PATH, "sha256": _p3_file_hash(path), "bytes": path.stat().st_size})
    return root, records


In [24]:
CORE_SOURCE_SHA256 = sector3_runtime_core_sha256(
    entrypoint=build_sector3_tables,
    namespace=globals(),
    dependency_versions={"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__},
)
PROCESSING_SIGNATURE = sector3_cache_signature(
    input_sha256=LINEAGE["input_sha256"],
    core_source_sha256=CORE_SOURCE_SHA256,
    config=CONFIG,
)
LINEAGE = {**LINEAGE, "core_source_sha256": CORE_SOURCE_SHA256}
SECTOR3_CACHE = io_load_cached_sector3(CONFIG, PROCESSING_SIGNATURE)

TABLES = (
    SECTOR3_CACHE["tables"]
    if SECTOR3_CACHE is not None
    else build_sector3_tables(
        metadata_clean=INPUTS["metadata_clean"],
        df_alignment_confirmed=INPUTS["df_alignment_confirmed"],
        video_manifest=INPUTS["video_manifest"],
        decode_audit=INPUTS["decode_audit"],
        config=CONFIG,
    )
)
TABLES = {**TABLES, "config_conflict_report": LINEAGE["config_conflict_report"].copy()}
SECTOR3_TRANSFORM_STATUS = "loaded existing transformations" if SECTOR3_CACHE is not None else "computed transformations"

labelled_frame_manifest = TABLES["labelled_frame_manifest"]
video_frame_manifest = TABLES["video_frame_manifest"]
clinical_taxonomy_report = TABLES["clinical_taxonomy_report"]
labelled_frame_bounds_report = TABLES["labelled_frame_bounds_report"]
frame_inventory_report = TABLES["frame_inventory_report"]
bbox_validation_report = TABLES["bbox_validation_report"]
bbox_validation_summary = TABLES["bbox_validation_summary"]
video_class_matrix = TABLES["video_class_matrix"]
video_class_counts = TABLES["video_class_counts"]
class_distribution_report = TABLES["class_distribution_report"]
video_class_support = TABLES["video_class_support"]

print("Sector 3:", SECTOR3_TRANSFORM_STATUS)
display(pd.DataFrame({"artifact": list(TABLES), "rows": [len(table) for table in TABLES.values()]}))

Sector 3: computed transformations


,artifact,rows
0,labelled_frame_manifest,47239
1,video_frame_manifest,117
2,clinical_taxonomy_report,14
3,labelled_frame_bounds_report,43
4,frame_inventory_report,7
5,bbox_validation_report,47239
6,bbox_validation_summary,5
7,video_class_matrix,43
8,video_class_counts,43
9,class_distribution_report,14


### 5. Inspect taxonomy, frame inventory, bbox status, and class coverage


In [25]:
display(clinical_taxonomy_report)
display(frame_inventory_report)
display(labelled_frame_bounds_report)
display(bbox_validation_summary)
display(class_distribution_report)
display(video_class_support)
display(video_class_matrix.head())
display(video_class_counts.head())
display(labelled_frame_manifest.head())

print("Storage:", SECTOR3_PERSISTED["status"])
print("Sector folder:", SECTOR3_PERSISTED["local_root"])
display(pd.DataFrame(SECTOR3_PERSISTED["artifacts"]))


,finding_class_normalized,expected_clinical_group,finding_class,clinical_group,original_annotation_count,retained_class_annotation_count,mapping_validation_passed
0,ampulla of vater,anatomical_landmark,Ampulla of Vater,anatomical_landmark,10,10,True
1,ileocecal valve,anatomical_landmark,Ileocecal valve,anatomical_landmark,4189,4189,True
2,pylorus,anatomical_landmark,Pylorus,anatomical_landmark,1529,1529,True
3,blood fresh,bleeding,Blood - fresh,bleeding,446,446,True
4,blood hematin,bleeding,Blood - hematin,bleeding,12,12,True
5,foreign body,foreign_body,Foreign Body,foreign_body,776,776,True
6,lymphangiectasia,lymphatic_lesion,Lymphangiectasia,lymphatic_lesion,592,592,True
7,erosion,mucosal_lesion,Erosion,mucosal_lesion,507,507,True
8,erythema,mucosal_lesion,Erythema,mucosal_lesion,159,159,True
9,ulcer,mucosal_lesion,Ulcer,mucosal_lesion,854,854,True


,frame_category,actual_frame_count,expected_frame_count,count_unit,difference_from_reference,reference_matches,validation_passed,reference_is_diagnostic
0,total_decoded_frames,4765114,4741504,decoded_frame,23610,False,True,True
1,original_annotation_rows,47239,<NA>,annotation,<NA>,<NA>,True,True
2,original_unique_source_frame_references,47229,47238,source_frame_reference,-9,False,True,True
3,accepted_annotation_rows,47239,<NA>,annotation,<NA>,<NA>,True,True
4,accepted_unique_decoded_frames,47229,<NA>,decoded_frame,<NA>,<NA>,True,True
5,source_frame_references_without_accepted_evidence,0,<NA>,source_frame_reference,<NA>,<NA>,True,True
6,decoded_frames_without_accepted_evidence,4717885,<NA>,decoded_frame,<NA>,<NA>,True,True


,video_key,labelled_frame_count,minimum_aligned_frame_index,maximum_aligned_frame_index,video_frame_count,out_of_bounds_frame_count,validation_passed
0,04a78ef00c5245e0,1292,887,48873,50986,0,True
1,0531325b64674948,1191,232,10351,16151,0,True
2,0728084c8da942d9,1499,4204,30209,48948,0,True
3,07c1fa15a20a4398,678,1353,38619,38620,0,True
4,131368cc17e44240,2034,1141,31698,31721,0,True
5,2fc3db471f9d44c0,127,1252,5511,76859,0,True
6,39960e5e099a45ca,237,9077,32922,32943,0,True
7,3ada4222967f421d,1002,27,40528,40529,0,True
8,3c8d5f0b90d7475d,1704,3381,45032,45033,0,True
9,4560e83f9afc4685,780,874,20367,35019,0,True


,bbox_validation_status,annotation_count
0,valid,4127
1,missing,43004
2,incomplete,0
3,invalid_geometry,0
4,out_of_image_bounds,108


,finding_class_normalized,clinical_group,original_annotation_count,original_unique_source_frame_count,retained_annotation_count,retained_unique_decoded_frame_count,retained_video_count,excluded_annotation_count,annotation_retention_percent,class_missing_after_alignment,rare_video_support
0,ampulla of vater,anatomical_landmark,10,10,10,10,1,0,100.0,False,True
1,angiectasia,vascular_lesion,866,866,866,866,6,0,100.0,False,False
2,blood fresh,bleeding,446,446,446,446,2,0,100.0,False,True
3,blood hematin,bleeding,12,12,12,12,1,0,100.0,False,True
4,erosion,mucosal_lesion,507,506,507,506,9,0,100.0,False,False
5,erythema,mucosal_lesion,159,159,159,159,3,0,100.0,False,False
6,foreign body,foreign_body,776,776,776,776,4,0,100.0,False,False
7,ileocecal valve,anatomical_landmark,4189,4189,4189,4189,34,0,100.0,False,False
8,lymphangiectasia,lymphatic_lesion,592,592,592,592,3,0,100.0,False,False
9,normal clean mucosa,normal_mucosa,34338,34338,34338,34338,37,0,100.0,False,False


...

,video_key,ampulla of vater,angiectasia,blood fresh,blood hematin,erosion,erythema,foreign body,ileocecal valve,lymphangiectasia,normal clean mucosa,polyp,pylorus,reduced mucosal view,ulcer
0,04a78ef00c5245e0,0,1,1,0,0,0,0,1,1,1,0,1,0,0
1,0531325b64674948,0,0,0,0,1,0,0,0,0,1,0,1,0,0
2,0728084c8da942d9,0,0,0,0,1,0,0,1,0,1,0,1,0,0
3,07c1fa15a20a4398,0,0,0,0,0,0,0,1,0,1,0,0,0,0
4,131368cc17e44240,0,0,0,0,0,0,0,1,0,1,1,1,0,0


,video_key,ampulla of vater,angiectasia,blood fresh,blood hematin,erosion,erythema,foreign body,ileocecal valve,lymphangiectasia,normal clean mucosa,polyp,pylorus,reduced mucosal view,ulcer
0,04a78ef00c5245e0,0,108,22,0,0,0,0,412,74,656,0,20,0,0
1,0531325b64674948,0,0,0,0,16,0,0,0,0,996,0,179,0,0
2,0728084c8da942d9,0,0,0,0,81,0,0,15,0,1399,0,4,0,0
3,07c1fa15a20a4398,0,0,0,0,0,0,0,10,0,668,0,0,0,0
4,131368cc17e44240,0,0,0,0,0,0,0,1261,0,617,55,101,0,0


,filename,video_id,frame_number,finding_category,finding_class,x1,y1,x2,y2,x3,...,image_path,video_path,annotation_status,label_source,source_frame_width,source_frame_height,bbox_within_image_bounds,bbox_spatially_usable,bbox_validation_status,bbox_bounds_policy
0,04a78ef00c5245e0_887.jpg,04a78ef00c5245e0,887,Luminal,Angiectasia,245.0,2.0,279.0,2.0,279.0,...,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,source_expert_annotation_alignment_confirmed,kvasir_capsule_expert_annotation,336,336,True,True,valid,normalized_xyxy_max_edge_may_equal_source_widt...
1,04a78ef00c5245e0_888.jpg,04a78ef00c5245e0,888,Luminal,Angiectasia,233.0,1.0,272.0,1.0,272.0,...,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,source_expert_annotation_alignment_confirmed,kvasir_capsule_expert_annotation,336,336,True,True,valid,normalized_xyxy_max_edge_may_equal_source_widt...
2,04a78ef00c5245e0_889.jpg,04a78ef00c5245e0,889,Luminal,Angiectasia,243.0,2.0,279.0,2.0,279.0,...,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,source_expert_annotation_alignment_confirmed,kvasir_capsule_expert_annotation,336,336,True,True,valid,normalized_xyxy_max_edge_may_equal_source_widt...
3,04a78ef00c5245e0_890.jpg,04a78ef00c5245e0,890,Luminal,Angiectasia,255.0,12.0,290.0,12.0,290.0,...,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,source_expert_annotation_alignment_confirmed,kvasir_capsule_expert_annotation,336,336,True,True,valid,normalized_xyxy_max_edge_may_equal_source_widt...
4,04a78ef00c5245e0_891.jpg,04a78ef00c5245e0,891,Luminal,Angiectasia,266.0,22.0,295.0,22.0,295.0,...,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,/content/drive/MyDrive/MMVQA_Clinical/data/raw...,source_expert_annotation_alignment_confirmed,kvasir_capsule_expert_annotation,336,336,True,True,valid,normalized_xyxy_max_edge_may_equal_source_widt...


Storage: unchanged
Sector folder: /content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/sector3


,bytes,relative_path,sha256
0,66088,configs/sector3_config_20261002T004948788764Z....,eef052b7a4d44182538d1dc2255e7ec9a38cb704376414...
1,93425,configs/sector3_config_20261008T001953882922Z....,c51c4fa39a7259cd3320f5c93096c0a1275868966a43ad...
2,1262133,reports/bbox_validation_report_20261002T004948...,9d358e2ed1d9b916825932f63cb154f60e639c19f7a2d4...
3,1973,reports/bbox_validation_report_20261002T004948...,3d6d5119cda3c7ab3e8b6eef6e0a365643564706b3bc94...
4,121,reports/bbox_validation_summary_20261002T00494...,cf0310534f6f94d7064c7fe85a098aabcdfe426f4b14bd...
5,453,reports/bbox_validation_summary_20261002T00494...,ed6603fbabcd841764b58c8ea208fa2bf98d24405a8ca2...
6,1229,reports/class_distribution_report_20261002T004...,54140e0795a424f0e35b7a8dc0253a5f10f4204b5d18a8...
7,1213,reports/class_distribution_report_20261002T004...,a9df1071952ba9aca04fcb1bdec40aedcc6adc53dcef01...
8,1156,reports/clinical_taxonomy_report_20261002T0049...,1a40b1f5ab5cc3f92f16b84d5955ab59d20456b5aecfe2...
9,940,reports/clinical_taxonomy_report_20261002T0049...,7f9d74d1ec81d18e95342aa468621cb78401800227a4e4...


### 6. Fetch GitHub and publish only changed Sector 3 artifacts


In [26]:
def _s3_pub_hash(path):
    with Path(path).open("rb") as source:
        return hashlib.file_digest(source, "sha256").hexdigest()


def _s3_pub_relative(value):
    """Accept only unambiguous catalog-relative POSIX paths."""
    value = str(value)
    path = PurePosixPath(value)
    if path.is_absolute() or ".." in path.parts or not path.parts or "\\" in value:
        raise ValueError(f"Unsafe publication path: {value!r}")
    if path.as_posix() != value or path.parts[0] not in {"configs", "results", "reports"}:
        raise ValueError(f"Publication path lies outside the Sector 3 artifact groups: {value!r}")
    return value


def _s3_pub_safe_target(repo, relative):
    target = repo / relative
    if target.is_symlink() or repo not in target.resolve().parents or any(
        parent.is_symlink() for parent in target.parents if parent != repo and repo in parent.parents
    ):
        raise RuntimeError("Publication targets cannot contain symlinks or leave the repository.")
    if target.exists() and not target.is_file():
        raise RuntimeError(f"Publication target is not a regular file: {relative}")
    return target


def _s3_pub_legacy_path(path, export):
    """Recognize only tracked artifacts generated by the former per-run notebook."""
    prefix = export + "/"
    if not isinstance(path, str) or not path.startswith(prefix):
        return False
    relative = path[len(prefix):]
    stamp = r"[0-9]{8}_[0-9]{6}(?:_[0-9]{6})?"
    if re.fullmatch(r"configs/sector3_state_" + stamp + r"\.json", relative):
        return True
    match = re.fullmatch(r"(results|reports)/(" + stamp + r")/([^/]+)", relative)
    if match is None:
        return False
    group, _, filename = match.groups()
    manifests = {"labelled_frame_manifest", "video_frame_manifest"}
    matrices = {"video_class_matrix", "video_class_counts"}
    reports = {"clinical_taxonomy_report", "labelled_frame_bounds_report", "frame_inventory_report",
               "bbox_validation_report", "bbox_validation_summary", "class_distribution_report",
               "video_class_support", "config_conflict_report"}
    if group == "results":
        return filename in ({name + ".parquet" for name in manifests}
                            | {name + suffix for name in matrices for suffix in (".csv", ".csv.gz")})
    generated_reports = {name + suffix for name in reports for suffix in (".csv", ".csv.gz")}
    generated_schemas = {name + "_schema.json" for name in manifests | matrices | reports}
    return filename in generated_reports | generated_schemas | {"summary.json", "executive_report.md"}


@contextlib.contextmanager
def _s3_pub_environment(settings, require_token=False):
    """Read a secret only for an actual push; keep it in the child environment."""
    environment = {
        key: value for key, value in os.environ.items()
        if not key.startswith("GIT_TRACE") and key not in {"GIT_CURL_VERBOSE", "GIT_ASKPASS", "SSH_ASKPASS"}
    }
    environment["GIT_TERMINAL_PROMPT"] = "0"
    secret = settings.get("token_secret_name", "GITHUB_TOKEN")
    local_test = settings.get("allow_local_test_remote") is True
    token = "" if local_test else os.environ.get(secret, "").strip()
    if require_token and not token and not local_test:
        try:
            from google.colab import userdata
            token = userdata.get(secret).strip()
        except Exception:
            raise RuntimeError(
                f"Enable Colab access to Secret {secret!r}, or provide it as an environment variable. "
                "No new commit was created."
            ) from None
        if not token:
            raise RuntimeError(f"Secret {secret!r} is empty. No new commit was created.")
    with tempfile.TemporaryDirectory(prefix="sector3-git-auth-") as directory:
        if token:
            askpass = Path(directory) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\nimport os, sys\n"
                "print(os.environ['S3_GIT_USERNAME'] if 'username' in sys.argv[1].lower() "
                "else os.environ['S3_GIT_TOKEN'])\n", encoding="utf-8",
            )
            askpass.chmod(0o700)
            environment.update({
                "GIT_ASKPASS": str(askpass), "S3_GIT_TOKEN": token,
                "S3_GIT_USERNAME": settings.get("github_username", "x-access-token"),
            })
        yield environment


def _s3_pub_git(repo, arguments, environment):
    completed = subprocess.run(
        ["git", "-C", str(repo), *arguments], env=environment, capture_output=True, text=True,
    )
    if completed.returncode:
        message = completed.stderr.strip() or completed.stdout.strip()
        token = environment.get("S3_GIT_TOKEN")
        if token:
            message = message.replace(token, "[REDACTED]")
        raise RuntimeError(f"Git {arguments[0]} failed: {message}")
    return completed.stdout.strip()


def _s3_pub_url(value, allow_local=False):
    value = str(value).rstrip("/").removesuffix(".git")
    if allow_local:
        return str(Path(value).resolve())
    if not re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", value):
        raise ValueError("GitHub origin must be a credential-free HTTPS owner/repository URL.")
    return value


def _s3_pub_receipt(path, receipt):
    """Install an operational receipt atomically, only after a publication action."""
    payload = json.dumps(receipt, sort_keys=True, indent=2, ensure_ascii=False, allow_nan=False) + "\n"
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,
                                     prefix=".sector3-receipt-", delete=False) as stream:
        temporary = Path(stream.name)
        stream.write(payload)
    try:
        if temporary.read_text(encoding="utf-8") != payload:
            raise RuntimeError("Publication receipt readback failed.")
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


def io_publish_sector3(
    persisted,
    config,
    *,
    push=None,
    dry_run=None,
):
    """Fetch; publish changed catalog content; leave identical reruns untouched."""
    root, records = io_checked_sector3_bundle(persisted, config)
    state = persisted["state"]
    content_id = state["content_id"]
    settings = config["github"]
    repo = Path(settings["local_repo_dir"]).expanduser().resolve()
    branch = str(settings["branch"])
    export = str(config["sector3"]["export_dir"])
    if export != "outputs/phase2/sector3":
        raise ValueError("Sector 3 publication destination must be outputs/phase2/sector3.")
    relative_records = [_s3_pub_relative(record["relative_path"]) for record in records]
    if len(relative_records) != len(set(relative_records)):
        raise RuntimeError("The publication catalog contains duplicate paths.")
    if "configs/sector3_publication_receipt.json" in relative_records:
        raise RuntimeError("The local publication receipt must not appear in the export catalog.")
    paths = [f"{export}/{relative}" for relative in relative_records]
    current_paths = set(relative_records)
    obsolete = sorted({_s3_pub_relative(value) for value in state.get("obsolete_paths", [])})
    if any(not value.startswith("results/") or value in current_paths for value in obsolete):
        raise RuntimeError("Obsolete paths must name only superseded current-result files.")
    obsolete_paths = [f"{export}/{relative}" for relative in obsolete]
    receipt_path = root / "configs/sector3_publication_receipt.json"
    if receipt_path.is_symlink():
        raise RuntimeError("The local publication receipt cannot be a symlink.")
    pending = json.loads(receipt_path.read_text(encoding="utf-8")) if receipt_path.exists() else {}
    if not isinstance(pending, dict):
        raise RuntimeError("The local publication receipt is not a JSON object.")
    push = (
        settings.get("push", True)
        if push is None
        else push
    )

    dry_run = (
        settings.get("dry_run", False)
        if dry_run is None
        else dry_run
    )

    if not isinstance(push, bool) or not isinstance(dry_run, bool):
        raise TypeError("push and dry_run must be booleans.")
    with _s3_pub_environment(settings) as environment:
        git = lambda *arguments: _s3_pub_git(repo, list(arguments), environment)
        if branch.startswith("-"):
            raise ValueError("Unsafe inherited Git branch name.")
        git("check-ref-format", "--branch", branch)
        if git("rev-parse", "--show-toplevel") != str(repo):
            raise RuntimeError("Configured local_repo_dir must be the repository root.")
        if git("symbolic-ref", "--short", "HEAD") != branch:
            raise RuntimeError("The checkout branch differs from inherited Git configuration.")
        local_test = settings.get("allow_local_test_remote") is True
        expected_remote = _s3_pub_url(settings["repo_url"], local_test)
        origins = git("remote", "get-url", "--all", "origin").splitlines()
        origins += git("remote", "get-url", "--push", "--all", "origin").splitlines()
        if not origins or any(_s3_pub_url(value, local_test) != expected_remote for value in origins):
            raise RuntimeError("Repository fetch/push origin differs from inherited configuration.")
        if git("status", "--porcelain", "--untracked-files=all"):
            raise RuntimeError("Git working tree must be clean before Sector 3 publication.")
        git("fetch", "origin", f"refs/heads/{branch}:refs/remotes/origin/{branch}")
        head = git("rev-parse", "HEAD")
        remote = git("rev-parse", f"origin/{branch}")
        ahead = git("rev-list", "--count", f"origin/{branch}..HEAD")
        owned_pending = (
            pending.get("commit") == head and pending.get("content_id") == content_id
            and pending.get("export_dir") == export and pending.get("branch") == branch
            and pending.get("status") in {"pending_push", "committed_not_pushed"}
        )
        retry_owned = head != remote and ahead == "1" and owned_pending
        legacy_pending = pending.get("legacy_removed_paths", []) if owned_pending else []
        if not isinstance(legacy_pending, list) or any(
            not _s3_pub_legacy_path(path, export) for path in legacy_pending
        ):
            raise RuntimeError("Pending receipt contains an unauthorized legacy deletion path.")
        if retry_owned:
            allowed_paths = set(paths + obsolete_paths + legacy_pending)
            changed = set(git("diff-tree", "--no-commit-id", "--name-only", "-r", head).splitlines())
            if not changed or not changed.issubset(allowed_paths):
                raise RuntimeError("Pending commit changes paths outside the verified Sector 3 catalog.")
            git("merge-base", "--is-ancestor", remote, head)
        elif head != remote:
            if ahead != "0":
                raise RuntimeError("Local branch has unrelated or divergent commits; publication stopped.")
            git("merge", "--ff-only", f"origin/{branch}")
            head = git("rev-parse", "HEAD")
        tracked = git("ls-files", "--", export).splitlines()
        legacy_current = [path for path in tracked if _s3_pub_legacy_path(path, export)]
        legacy_authorized = sorted(set(legacy_current + legacy_pending))
        allowed_paths = set(paths + obsolete_paths + legacy_authorized)
        targets = [_s3_pub_safe_target(repo, path) for path in paths]
        outdated = [
            (record, relative, target) for record, relative, target in zip(records, paths, targets)
            if not target.exists() or _s3_pub_hash(target) != record["sha256"]
        ]
        removals = [(_s3_pub_safe_target(repo, path), path)
                    for path in sorted(set(obsolete_paths + legacy_current))]
        removals = [(target, path) for target, path in removals if target.exists()]
        if retry_owned and (outdated or removals):
            raise RuntimeError("The owned pending commit no longer matches the checked Sector 3 catalog.")
        immutable_conflicts = [
            relative for record, relative, target in outdated
            if target.exists() and record["relative_path"].split("/", 1)[0] in {"configs", "reports"}
            and record["relative_path"] != "configs/sector3_state.json"
        ]
        if immutable_conflicts:
            raise FileExistsError(f"An existing config/report version differs from its catalog: {immutable_conflicts[:5]}")
        if not outdated and not removals and not retry_owned:
            return {"status": "unchanged", "content_id": content_id, "commit": head,
                    "branch": branch, "export_dir": export, "artifact_count": len(records),
                    "changed_artifact_count": 0}
        if dry_run:
            return {"status": "dry_run", "content_id": content_id, "commit": head,
                    "branch": branch, "export_dir": export, "artifact_count": len(records),
                    "changed_artifact_count": len(outdated) + len(removals)}
        if retry_owned and not push:
            return dict(pending)
        for author_field in ("author_name", "author_email"):
            if not str(settings.get(author_field, "")).strip():
                raise ValueError(f"Inherited Git setting {author_field} must be nonempty.")
        # A missing push credential must fail before copy/staging/commit mutations.
        with _s3_pub_environment(settings, require_token=push) as publication_environment:
            publish_git = lambda *arguments: _s3_pub_git(repo, list(arguments), publication_environment)
            if not retry_owned:
                changed_paths = []
                for record, relative, target in outdated:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with tempfile.NamedTemporaryFile(dir=target.parent, delete=False, prefix=".sector3-") as stream:
                        temporary = Path(stream.name)
                    try:
                        shutil.copyfile(root / record["relative_path"], temporary)
                        if _s3_pub_hash(temporary) != record["sha256"]:
                            raise RuntimeError(f"Publication copy readback failed: {relative}")
                        temporary.replace(target)
                    finally:
                        temporary.unlink(missing_ok=True)
                    changed_paths.append(relative)
                for target, relative in removals:
                    target.unlink()
                    changed_paths.append(relative)
                publish_git("add", "--all", "--", *changed_paths)
                staged = set(publish_git("diff", "--cached", "--name-only").splitlines())
                if not staged.issubset(allowed_paths):
                    raise RuntimeError("Unrelated staged paths detected; stopped before commit.")
                if not staged:
                    raise RuntimeError("Checksummed artifact differences produced no Git changes.")
                publish_git("-c", f"user.name={settings['author_name']}", "-c", f"user.email={settings['author_email']}",
                            "commit", "-m", f"Phase 2 Sector 3: update manifest and audits ({content_id[:12]})")
                head = publish_git("rev-parse", "HEAD")
            receipt = {"status": "pending_push" if push else "committed_not_pushed",
                       "content_id": content_id, "commit": head, "branch": branch,
                       "export_dir": export, "artifact_count": len(records),
                       "legacy_removed_paths": legacy_authorized,
                       "saved_at_utc": datetime.now(timezone.utc).isoformat()}
            _s3_pub_receipt(receipt_path, receipt)
            if push:
                try:
                    publish_git("push", "origin", f"HEAD:refs/heads/{branch}")
                except RuntimeError as error:
                    raise RuntimeError(
                        f"Sector 3 was committed as {head}; push failed. Keep this catalog and rerun "
                        f"io_publish_sector3 to retry its owned commit. {error}"
                    ) from None
                receipt["status"] = "published"
                _s3_pub_receipt(receipt_path, receipt)
            return receipt


In [27]:
SECTOR3_PERSISTED = io_persist_sector3(
    tables=TABLES,
    config=CONFIG,
    lineage=LINEAGE,
    processing_signature=PROCESSING_SIGNATURE,
)

print("Storage:", SECTOR3_PERSISTED["status"])
print("Sector folder:", SECTOR3_PERSISTED["local_root"])

display(
    pd.DataFrame(
        SECTOR3_PERSISTED["artifacts"]
    )
)

Storage: updated
Sector folder: /content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/sector3


,bytes,relative_path,sha256
0,66088,configs/sector3_config_20261002T004948788764Z....,eef052b7a4d44182538d1dc2255e7ec9a38cb704376414...
1,93425,configs/sector3_config_20261008T001953882922Z....,c51c4fa39a7259cd3320f5c93096c0a1275868966a43ad...
2,93425,configs/sector3_config_20261008T003651096340Z....,caf919b46e9b83ba3ace0640b6191e61cf5b8682a72dfa...
3,1262133,reports/bbox_validation_report_20261002T004948...,9d358e2ed1d9b916825932f63cb154f60e639c19f7a2d4...
4,1973,reports/bbox_validation_report_20261002T004948...,3d6d5119cda3c7ab3e8b6eef6e0a365643564706b3bc94...
5,121,reports/bbox_validation_summary_20261002T00494...,cf0310534f6f94d7064c7fe85a098aabcdfe426f4b14bd...
6,453,reports/bbox_validation_summary_20261002T00494...,ed6603fbabcd841764b58c8ea208fa2bf98d24405a8ca2...
7,1229,reports/class_distribution_report_20261002T004...,54140e0795a424f0e35b7a8dc0253a5f10f4204b5d18a8...
8,1213,reports/class_distribution_report_20261002T004...,a9df1071952ba9aca04fcb1bdec40aedcc6adc53dcef01...
9,1156,reports/clinical_taxonomy_report_20261002T0049...,1a40b1f5ab5cc3f92f16b84d5955ab59d20456b5aecfe2...


### 7. Fetch GitHub and publish the completed Sector 3 run



In [28]:
PUBLICATION_OPTIONS = {
    "push": True,
    "dry_run": False,
}

SECTOR3_PUBLICATION = io_publish_sector3(
    SECTOR3_PERSISTED,
    CONFIG,
    **PUBLICATION_OPTIONS,
)

display(pd.DataFrame.from_records([SECTOR3_PUBLICATION]))

,status,content_id,commit,branch,export_dir,artifact_count,legacy_removed_paths,saved_at_utc
0,published,5d6312664ccd0c6fce0f9ab174a0b1b8c549607a75cc48...,ed41a67c8af5b193204d60b3b210ece1404cd4cb,main,outputs/phase2/sector3,33,[],2026-10-08T00:36:52.096702+00:00
